# Symbolic verification of the `curl_*_force_*` diagnostics

This notebook independently re-derives, with `sympy`, the true spherical curl
of each force in `Diagnostics_Curl_Momentum.F90` from that force's own
baseline formula (as coded in `Diagnostics_Linear_Forces.F90`,
`Diagnostics_Curl_Momentum.F90`'s `Grad_Viscous_Force`, etc.), and then checks
the *actual* Fortran expression coded for the corresponding `curl_*` quantity
against that derivation term-by-term.

For every force below, `report()` prints `MATCH` when
`sympy.simplify(true_curl - code_expression)` is identically zero, and prints
the (non-zero) residual otherwise.

Convention: `theta` is the polar angle from the rotation axis and `phi` is
the azimuthal angle, matching Rayleigh's spherical coordinate convention.
The Fortran-side names used in comparisons below (`buffer(PSI,...)`,
`DDBUFF(PSI,...)`, `one_over_r(r)`, `costheta(t)`, ...) are written as plain
`sympy` symbols with those exact string names, so the printed "code"
expressions can be read directly against the `.F90` source.

## Setup

Standard spherical curl:
$$(\nabla\times F)_r=\frac{1}{r\sin\theta}\left[\partial_\theta(F_\phi\sin\theta)-\partial_\phi F_\theta\right]$$
$$(\nabla\times F)_\theta=\frac{1}{r}\left[\frac{1}{\sin\theta}\partial_\phi F_r-\partial_r(rF_\phi)\right]$$
$$(\nabla\times F)_\phi=\frac{1}{r}\left[\partial_r(rF_\theta)-\partial_\theta F_r\right]$$

In [1]:

import re
import sympy as sp
sp.init_printing(use_latex='mathjax')

r, theta, phi = sp.symbols('r theta phi', positive=True)

def curl(Fr, Ft, Fp):
    # Standard spherical curl. theta=polar angle from axis, phi=azimuth.
    curl_r = (1/(r*sp.sin(theta))) * (sp.diff(Fp*sp.sin(theta), theta) - sp.diff(Ft, phi))
    curl_t = (1/r) * (sp.diff(Fr, phi)/sp.sin(theta) - sp.diff(r*Fp, r))
    curl_p = (1/r) * (sp.diff(r*Ft, r) - sp.diff(Fr, theta))
    return sp.simplify(curl_r), sp.simplify(curl_t), sp.simplify(curl_p)

def report(name, computed, coded):
    diff = sp.simplify(sp.expand_trig(sp.expand(computed - coded)))
    ok = diff == 0
    print(f"{name}: {'MATCH' if ok else 'MISMATCH'}")
    if not ok:
        print(f"   true - code = {diff}")
    return ok

# ---------------------------------------------------------------------
# Fortran-style pretty printer: same macro-aware line-splitting logic
# used to generate the actual qty(PSI) = ... assignments in
# Diagnostics_Curl_Momentum.F90 (DDBUFF -> d2buffer%p3a and PSI -> k,r,t
# expand under the C preprocessor, inflating the true line length beyond
# what the raw source shows -- so wrapping must budget for that expansion,
# not the raw string length).
# ---------------------------------------------------------------------
def _expand_macros(s):
    s2 = re.sub(r'\bDDBUFF\b', 'd2buffer%p3a', s)
    s2 = re.sub(r'\bPSI\b', 'k,r,t', s2)
    return s2

def _expanded_len(s):
    return len(_expand_macros(s))

def fortran_lines(expr, lhs='qty(PSI) = ', indent=' '*16, maxlen=126):
    terms = sp.Add.make_args(sp.expand(expr))
    pieces = []
    for t in terms:
        s = sp.sstr(t)
        if s.startswith('-'):
            pieces.append(('-', s[1:].strip()))
        else:
            pieces.append(('+', s))

    lines = []
    cur = None
    for idx, (sign, body) in enumerate(pieces):
        if idx == 0:
            cur = indent + lhs + (('-' if sign == '-' else '') + body)
        else:
            candidate = cur + f' {sign} ' + body
            if _expanded_len(candidate + ' &') <= maxlen:
                cur = candidate
            else:
                lines.append(cur + ' &')
                cur = indent + f'{sign} ' + body
    lines.append(cur)
    return '\n'.join(lines)


## 1. Coriolis force

Baseline (`Diagnostics_Linear_Forces.F90`, `Compute_Coriolis_Force`), with
$C=$`ref%Coriolis_Coeff` and $\rho=$`ref%density(r)`:

$$F_r = C\rho\sin\theta\,v_\phi,\qquad F_\theta = C\rho\cos\theta\,v_\phi,\qquad
F_\phi = -C\rho(\cos\theta\,v_\theta + \sin\theta\,v_r)$$

In [2]:

C = sp.Symbol('C')  # ref%Coriolis_Coeff
rho = sp.Function('rho')(r)
dlnrho = sp.diff(rho, r)/rho

vr = sp.Function('v_r')(r, theta, phi)
vt = sp.Function('v_t')(r, theta, phi)
vp = sp.Function('v_p')(r, theta, phi)

dvrdr, dvrdt, dvrdp = sp.diff(vr,r), sp.diff(vr,theta), sp.diff(vr,phi)
dvtdr, dvtdt, dvtdp = sp.diff(vt,r), sp.diff(vt,theta), sp.diff(vt,phi)
dvpdr, dvpdt, dvpdp = sp.diff(vp,r), sp.diff(vp,theta), sp.diff(vp,phi)

Fr_cor = C*rho*sp.sin(theta)*vp
Ft_cor = C*rho*sp.cos(theta)*vp
Fp_cor = -C*rho*(sp.cos(theta)*vt + sp.sin(theta)*vr)

Fr_cor, Ft_cor, Fp_cor


(C⋅ρ(r)⋅vₚ(r, θ, φ)⋅sin(θ), C⋅ρ(r)⋅vₚ(r, θ, φ)⋅cos(θ), -C⋅(vᵣ(r, θ, φ)⋅sin(θ) 
+ vₜ(r, θ, φ)⋅cos(θ))⋅ρ(r))

### True curl (symbolic)

In [3]:

cr_cor, ct_cor, cp_cor = curl(Fr_cor, Ft_cor, Fp_cor)
cr_cor, ct_cor, cp_cor


⎛  ⎛                                                                          
⎜  ⎜                                                                          
⎜  ⎜                                               vₜ(r, θ, φ)          ∂     
⎜C⋅⎜-2⋅vᵣ(r, θ, φ)⋅cos(θ) + 2⋅vₜ(r, θ, φ)⋅sin(θ) - ─────────── - sin(θ)⋅──(vᵣ(
⎜  ⎝                                                  sin(θ)            ∂θ    
⎜─────────────────────────────────────────────────────────────────────────────
⎝                                                                   r         

                                     ∂              ⎞                         
                                     ──(vₚ(r, θ, φ))⎟                         
                   ∂                 ∂φ             ⎟         ⎛               
r, θ, φ)) - cos(θ)⋅──(vₜ(r, θ, φ)) - ───────────────⎟⋅ρ(r)  C⋅⎜r⋅(vᵣ(r, θ, φ)⋅
                   ∂θ                     tan(θ)    ⎠         ⎝               
───────────────────────────────────────────────────

### Compare against the current Fortran (`Compute_Curl_Coriolis_Force`, lines 614-654)

In [4]:

oor = 1/r
csc = 1/sp.sin(theta)
cot = sp.cos(theta)/sp.sin(theta)
cos_ = sp.cos(theta)
sin_ = sp.sin(theta)

# curl_coriolis_force_r
code_r_cor = - C*rho*oor*(-sin_*vt + cot*cos_*vt + cos_*dvtdt + 2*cos_*vr + sin_*dvrdt + cot*dvpdp)
report("curl_coriolis_force_r", cr_cor, code_r_cor)

# curl_coriolis_force_theta
code_t_cor = C*rho*(oor*(dvpdp + cos_*vt + sin_*vr) + cos_*dvtdr + sin_*dvrdr + dlnrho*cos_*vt + dlnrho*sin_*vr)
report("curl_coriolis_force_theta", ct_cor, code_t_cor)

# curl_coriolis_force_phi
code_p_cor = C*rho*(dlnrho*cos_*vp + cos_*dvpdr - oor*sin_*dvpdt)
report("curl_coriolis_force_phi", cp_cor, code_p_cor)


curl_coriolis_force_r: MATCH
curl_coriolis_force_theta: MATCH
curl_coriolis_force_phi: MATCH


True

## 2. Buoyancy force

Baseline (`Diagnostics_Linear_Forces.F90`, `Compute_Buoyancy_Force`) is purely
radial, with $B_c(r)=$`ref%Buoyancy_Coeff(r)` and $T=$`buffer(PSI,tvar)`:

$$F_r = B_c(r)\,(T - T_0(r)),\qquad F_\theta = F_\phi = 0$$

so $(\nabla\times F)_r \equiv 0$ (matches the code, which has no
`curl_buoyancy_force_r`).

In [5]:

Bc = sp.Function('Bc')(r)
T  = sp.Function('T')(r, theta, phi)
T0 = sp.Function('T0')(r)
dtdt, dtdp = sp.diff(T, theta), sp.diff(T, phi)

Fr_b = Bc*(T - T0)
Ft_b = sp.Integer(0)
Fp_b = sp.Integer(0)

cr_b, ct_b, cp_b = curl(Fr_b, Ft_b, Fp_b)
print("curl_r (true):", cr_b, " -- always 0, as expected")
ct_b, cp_b


curl_r (true): 0  -- always 0, as expected


⎛      ∂                      ∂              ⎞
⎜Bc(r)⋅──(T(r, θ, φ))  -Bc(r)⋅──(T(r, θ, φ)) ⎟
⎜      ∂φ                     ∂θ             ⎟
⎜────────────────────, ──────────────────────⎟
⎝      r⋅sin(θ)                  r           ⎠

### Compare against the current Fortran (`Compute_Curl_Buoyancy_Force`, lines 278-305)

In [6]:

code_t_b = Bc * csc * oor * dtdp
report("curl_buoyancy_force_theta", ct_b, code_t_b)

code_p_b = -Bc * oor * dtdt
report("curl_buoyancy_force_phi", cp_b, code_p_b)


curl_buoyancy_force_theta: MATCH
curl_buoyancy_force_phi: MATCH


True

## 3. Pressure force

Baseline (`Diagnostics_Linear_Forces.F90`, `Compute_Pressure_Force`), with
`pfactor = ref%dpdr_w_term(r)/ref%density(r)` (the code's own comment notes
`pfactor` is treated as constant for this curl derivation) and
$\lambda(r)=$`ref%dlnrho(r)`:

$$F_r = -\text{pfactor}\,\partial_r(P-P_0) + \text{pfactor}\,\lambda(r)(P-P_0),\qquad
F_\theta = -\frac{\text{pfactor}}{r}\partial_\theta P,\qquad
F_\phi = -\frac{\text{pfactor}}{r\sin\theta}\partial_\phi P$$

In [7]:

pfactor = sp.Symbol('p_f')
dlnrho = sp.Function('dlnrho')(r)
P  = sp.Function('P')(r, theta, phi)
P0 = sp.Function('P0')(r)
dpdt, dpdp = sp.diff(P, theta), sp.diff(P, phi)

Fr_p = -pfactor*(sp.diff(P, r) - sp.diff(P0, r)) + pfactor*dlnrho*(P - P0)
Ft_p = -pfactor*dpdt/r
Fp_p = -pfactor*dpdp/(r*sp.sin(theta))

cr_p, ct_p, cp_p = curl(Fr_p, Ft_p, Fp_p)
print("curl_r (true):", cr_p, " -- always 0, as expected")
ct_p, cp_p


curl_r (true): 0  -- always 0, as expected


⎛              ∂                              ∂              ⎞
⎜p_f⋅dlnrho(r)⋅──(P(r, θ, φ))  -p_f⋅dlnrho(r)⋅──(P(r, θ, φ)) ⎟
⎜              ∂φ                             ∂θ             ⎟
⎜────────────────────────────, ──────────────────────────────⎟
⎝          r⋅sin(θ)                          r               ⎠

### Compare against the current Fortran (`Compute_Curl_Pressure_Force`, lines 693-721)

`curl_pressure_force_theta` needs the $\phi$-derivative of $P$ (`dpdp`), while
`curl_pressure_force_phi` needs the $\theta$-derivative (`dpdt`) — that
asymmetry is correct, not a typo (it falls straight out of the curl
formula).

In [8]:

code_t_p = pfactor * oor * csc * dlnrho * dpdp
report("curl_pressure_force_theta", ct_p, code_t_p)

code_p_p = -pfactor * oor * dlnrho * dpdt
report("curl_pressure_force_phi", cp_p, code_p_p)


curl_pressure_force_theta: MATCH
curl_pressure_force_phi: MATCH


True

## 4. Viscous force

`Grad_Viscous_Force` computes first derivatives of the viscous force itself
(stored via the `VFDBUFF`/`vforce_buffer` arrays as a generic vector field
$(v\!f_r, v\!f_\theta, v\!f_\phi)$) — so no extra baseline substitution is
needed here, `curl_viscous_force_*` is simply the generic spherical curl of
that vector field.

In [9]:

vfr = sp.Function('vfr')(r, theta, phi)
vft = sp.Function('vft')(r, theta, phi)
vfp = sp.Function('vfp')(r, theta, phi)

cr_v, ct_v, cp_v = curl(vfr, vft, vfp)
cr_v, ct_v, cp_v


⎛                                  ∂                                          
⎜                                  ──(vft(r, θ, φ))                           
⎜vfp(r, θ, φ)   ∂                  ∂φ                    ∂                    
⎜──────────── + ──(vfp(r, θ, φ)) - ────────────────  - r⋅──(vfp(r, θ, φ)) - vf
⎜   tan(θ)      ∂θ                      sin(θ)           ∂r                   
⎜──────────────────────────────────────────────────, ─────────────────────────
⎝                        r                                                    

             ∂                                                                
             ──(vfr(r, θ, φ))                                                 
             ∂φ                  ∂                                 ∂          
p(r, θ, φ) + ────────────────  r⋅──(vft(r, θ, φ)) + vft(r, θ, φ) - ──(vfr(r, θ
                  sin(θ)         ∂r                                ∂θ         
─────────────────────────────, ────────────────────

### Compare against the current Fortran (`Compute_Curl_Viscous_Force`, lines 732-775)

In [10]:

code_r_v = oor*(sp.diff(vfp, theta) + cot*vfp - csc*sp.diff(vft, phi))
report("curl_viscous_force_r", cr_v, code_r_v)

code_t_v = oor*(csc*sp.diff(vfr, phi) - vfp) - sp.diff(vfp, r)
report("curl_viscous_force_theta", ct_v, code_t_v)

code_p_v = sp.diff(vft, r) + oor*(vft - sp.diff(vfr, theta))
report("curl_viscous_force_phi", cp_v, code_p_v)


curl_viscous_force_r: MATCH
curl_viscous_force_theta: MATCH
curl_viscous_force_phi: MATCH


True

## 5. Advection force $\rho\,(v\cdot\nabla)v$ (`v_grad_v`)

Baseline (`Diagnostics_Inertial_Forces.F90`, and the underlying
`ADotGradB_3D3D` dispatch used at the `v_grad_v` call site), with
$\rho=$`ref%density(r)` and the standard spherical $(v\cdot\nabla)v$:

$$(v\cdot\nabla v)_r = v_r\partial_r v_r + \frac{v_\theta}{r}\partial_\theta v_r
   + \frac{v_\phi}{r\sin\theta}\partial_\phi v_r - \frac{v_\theta^2+v_\phi^2}{r}$$
$$(v\cdot\nabla v)_\theta = v_r\partial_r v_\theta + \frac{v_\theta}{r}\partial_\theta v_\theta
   + \frac{v_\phi}{r\sin\theta}\partial_\phi v_\theta + \frac{v_rv_\theta}{r} - \frac{v_\phi^2\cot\theta}{r}$$
$$(v\cdot\nabla v)_\phi = v_r\partial_r v_\phi + \frac{v_\theta}{r}\partial_\theta v_\phi
   + \frac{v_\phi}{r\sin\theta}\partial_\phi v_\phi + \frac{v_rv_\phi}{r} + \frac{v_\theta v_\phi\cot\theta}{r}$$

$F = \rho\,(v\cdot\nabla)v$ is what is curled below.

In [11]:

vgv_r = vr*dvrdr + vt*oor*dvrdt + vp*oor*csc*dvrdp - (vt**2+vp**2)*oor
vgv_t = vr*dvtdr + vt*oor*dvtdt + vp*oor*csc*dvtdp + vr*vt*oor - vp**2*cot*oor
vgv_p = vr*dvpdr + vt*oor*dvpdt + vp*oor*csc*dvpdp + vr*vp*oor + vt*vp*cot*oor

Fr_a = rho*vgv_r
Ft_a = rho*vgv_t
Fp_a = rho*vgv_p

cr_a, ct_a, cp_a = curl(Fr_a, Ft_a, Fp_a)
cr_a, ct_a, cp_a


⎛                                                                             
⎜                                                                             
⎜                                                                             
⎜⎛  ⎛               2                                                ⎞        
⎜⎜  ⎜              ∂                  ∂               ∂              ⎟    2   
⎜⎜r⋅⎜vᵣ(r, θ, φ)⋅─────(vₚ(r, θ, φ)) + ──(vₚ(r, θ, φ))⋅──(vᵣ(r, θ, φ))⎟⋅sin (θ)
⎜⎜  ⎝            ∂θ ∂r                ∂r              ∂θ             ⎠        
⎜⎝                                                                            
⎜─────────────────────────────────────────────────────────────────────────────
⎜                                                                             
⎝                                                                             

                                                                              
                                                   

### Fortranize: substitute sympy `Derivative`/`Function` objects with symbols
named exactly like the Fortran buffer accessors, so the printed result can be
read straight off `Diagnostics_Curl_Momentum.F90`.

The substitution list is applied to $(\nabla\times F)_r,\ (\nabla\times F)_\theta,\
(\nabla\times F)_\phi$ above, and a completeness check confirms no
`Derivative`/`Function` objects are left over (i.e. the substitution was
lossless) before trusting the printed Fortran-symbol form.

In [12]:

fields = {'vr': vr, 'vt': vt, 'vp': vp}
field_buffer_name = {'vr': 'vr', 'vt': 'vtheta', 'vp': 'vphi'}   # buffer(PSI,<name>)
varsym = {'r': r, 't': theta, 'p': phi}

subs_list = []

# second derivatives first (must precede first-derivative substitution, since
# first-derivative patterns are sub-expressions of second derivatives)
pairs = [('r','r'), ('r','t'), ('r','p'), ('t','t'), ('t','p'), ('p','p')]
for fshort, ffunc in fields.items():
    for a, b in pairs:
        name = f'DDBUFF(PSI,d{fshort}d{a}d{b})'
        subs_list.append((sp.diff(ffunc, varsym[a], varsym[b]), sp.Symbol(name)))

# first derivatives
for fshort, ffunc in fields.items():
    for a in ['r','t','p']:
        name = f'buffer(PSI,d{fshort}d{a})'
        subs_list.append((sp.diff(ffunc, varsym[a]), sp.Symbol(name)))

# bare field values
for fshort, ffunc in fields.items():
    name = f'buffer(PSI,{field_buffer_name[fshort]})'
    subs_list.append((ffunc, sp.Symbol(name)))

# density: substitute the derivative FIRST (rho' = dlnrho*rho), then rho itself
RHO = sp.Symbol('ref%density(r)')
DLNRHO = sp.Symbol('ref%dlnrho(r)')
subs_list = [(sp.diff(rho, r), DLNRHO*RHO)] + subs_list + [(rho, RHO)]

# metric factors
OOR = sp.Symbol('one_over_r(r)')
CSC = sp.Symbol('csctheta(t)')
COT = sp.Symbol('cottheta(t)')
COS_ = sp.Symbol('costheta(t)')
SIN_ = sp.Symbol('sintheta(t)')
metric_subs = [(oor, OOR), (csc, CSC), (cot, COT), (cos_, COS_),
               (1/sp.tan(theta), COT), (sp.tan(theta), 1/COT)]

def fortranize(expr):
    e = expr.subs(subs_list)
    e = sp.expand(e)
    e = e.subs(metric_subs)
    return e

fr_out = fortranize(cr_a)
ft_out = fortranize(ct_a)
fp_out = fortranize(cp_a)

for name, e in [("curl_v_grad_v_r", fr_out), ("curl_v_grad_v_theta", ft_out), ("curl_v_grad_v_phi", fp_out)]:
    residual_atoms = [a for a in e.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
    print(f"{name}: {'CLEAN (fully expressed in Fortran symbols)' if not residual_atoms else 'LEFTOVER: '+str(residual_atoms)}")


curl_v_grad_v_r: CLEAN (fully expressed in Fortran symbols)
curl_v_grad_v_theta: CLEAN (fully expressed in Fortran symbols)
curl_v_grad_v_phi: CLEAN (fully expressed in Fortran symbols)


### `curl_v_grad_v_r` (Fortran-symbol form)

This is exactly the expression now coded at
`Diagnostics_Curl_Momentum.F90`'s `curl_v_grad_v_r` block (17 additive terms,
matching one-for-one with the lines of that `qty(PSI) = ...` assignment).

In [13]:
print(fortran_lines(fr_out))

                qty(PSI) = DDBUFF(PSI,dvpdrdt)*buffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                + DDBUFF(PSI,dvpdtdt)*buffer(PSI,vtheta)*one_over_r(r)**2*ref%density(r) &
                + buffer(PSI,dvpdr)*buffer(PSI,dvrdt)*one_over_r(r)*ref%density(r) &
                + buffer(PSI,dvpdt)*buffer(PSI,dvtdt)*one_over_r(r)**2*ref%density(r) &
                + buffer(PSI,dvpdt)*buffer(PSI,vr)*one_over_r(r)**2*ref%density(r) &
                + buffer(PSI,dvrdt)*buffer(PSI,vphi)*one_over_r(r)**2*ref%density(r) &
                - buffer(PSI,vphi)*buffer(PSI,vtheta)*one_over_r(r)**2*ref%density(r) &
                + DDBUFF(PSI,dvpdtdp)*buffer(PSI,vphi)*csctheta(t)*one_over_r(r)**2*ref%density(r) &
                + buffer(PSI,dvpdp)*buffer(PSI,dvpdt)*csctheta(t)*one_over_r(r)**2*ref%density(r) &
                - DDBUFF(PSI,dvtdpdp)*buffer(PSI,vphi)*csctheta(t)**2*one_over_r(r)**2*ref%density(r) &
                - DDBUFF(PSI,dvtdrdp)*buffer(PSI,vr)*csctheta(t)*one_over_r(r)

### `curl_v_grad_v_theta` (Fortran-symbol form)

In [14]:
print(fortran_lines(ft_out))

                qty(PSI) = -DDBUFF(PSI,dvpdrdr)*buffer(PSI,vr)*ref%density(r) &
                - buffer(PSI,dvpdr)*buffer(PSI,dvrdr)*ref%density(r) &
                - DDBUFF(PSI,dvpdrdt)*buffer(PSI,vtheta)*one_over_r(r)*ref%density(r) &
                - buffer(PSI,dvpdr)*buffer(PSI,vr)*ref%density(r)*ref%dlnrho(r) &
                - buffer(PSI,dvpdt)*buffer(PSI,dvtdr)*one_over_r(r)*ref%density(r) &
                - buffer(PSI,dvrdr)*buffer(PSI,vphi)*one_over_r(r)*ref%density(r) &
                - 2*buffer(PSI,dvpdr)*buffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                + DDBUFF(PSI,dvrdpdp)*buffer(PSI,vphi)*csctheta(t)**2*one_over_r(r)**2*ref%density(r) &
                + DDBUFF(PSI,dvrdrdp)*buffer(PSI,vr)*csctheta(t)*one_over_r(r)*ref%density(r) &
                + DDBUFF(PSI,dvrdtdp)*buffer(PSI,vtheta)*csctheta(t)*one_over_r(r)**2*ref%density(r) &
                + buffer(PSI,dvpdp)*buffer(PSI,dvrdp)*csctheta(t)**2*one_over_r(r)**2*ref%density(r) &
                + bu

### `curl_v_grad_v_phi` (Fortran-symbol form)

In [15]:
print(fortran_lines(fp_out))

                qty(PSI) = DDBUFF(PSI,dvtdrdr)*buffer(PSI,vr)*ref%density(r) &
                + buffer(PSI,dvrdr)*buffer(PSI,dvtdr)*ref%density(r) &
                + DDBUFF(PSI,dvtdrdt)*buffer(PSI,vtheta)*one_over_r(r)*ref%density(r) &
                + buffer(PSI,dvrdr)*buffer(PSI,vtheta)*one_over_r(r)*ref%density(r) &
                + buffer(PSI,dvtdr)*buffer(PSI,dvtdt)*one_over_r(r)*ref%density(r) &
                + buffer(PSI,dvtdr)*buffer(PSI,vr)*ref%density(r)*ref%dlnrho(r) &
                - DDBUFF(PSI,dvrdrdt)*buffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                - DDBUFF(PSI,dvrdtdt)*buffer(PSI,vtheta)*one_over_r(r)**2*ref%density(r) &
                - buffer(PSI,dvrdr)*buffer(PSI,dvrdt)*one_over_r(r)*ref%density(r) &
                - buffer(PSI,dvrdt)*buffer(PSI,dvtdt)*one_over_r(r)**2*ref%density(r) &
                + 2*buffer(PSI,dvpdt)*buffer(PSI,vphi)*one_over_r(r)**2*ref%density(r) &
                + 2*buffer(PSI,dvtdr)*buffer(PSI,vr)*one_over_r(r)*ref%d

### `curl_v_grad_v_abs`

Coded as $\rho(r)\sqrt{\hat r^2+\hat\theta^2+\hat\phi^2}$ where $\hat r,\hat\theta,\hat\phi$
are the `curl_v_grad_v_r/theta/phi` formulas above with the common factor
`ref%density(r)` divided out. Verify that factoring is exact (remainder is
identically zero for each component) before trusting the `vgv_abs_r/t/p`
local temporaries in the code.

In [16]:

vgv_abs_r = sp.simplify(sp.expand(fr_out)/RHO)
vgv_abs_t = sp.simplify(sp.expand(ft_out)/RHO)
vgv_abs_p = sp.simplify(sp.expand(fp_out)/RHO)

for name, full, bracket in [("r", fr_out, vgv_abs_r), ("theta", ft_out, vgv_abs_t), ("phi", fp_out, vgv_abs_p)]:
    residual = sp.simplify(full - RHO*bracket)
    print(f"{name}: density factors out cleanly = {residual == 0}")


r: density factors out cleanly = True
theta: density factors out cleanly = True
phi: density factors out cleanly = True


`vgv_abs_r` (Fortran-symbol form, `ref%density(r)` divided out)

In [17]:
print(fortran_lines(vgv_abs_r, lhs='vgv_abs_r = '))

                vgv_abs_r = DDBUFF(PSI,dvpdrdt)*buffer(PSI,vr)*one_over_r(r) &
                + DDBUFF(PSI,dvpdtdt)*buffer(PSI,vtheta)*one_over_r(r)**2 &
                + buffer(PSI,dvpdr)*buffer(PSI,dvrdt)*one_over_r(r) &
                + buffer(PSI,dvpdt)*buffer(PSI,dvtdt)*one_over_r(r)**2 &
                + buffer(PSI,dvpdt)*buffer(PSI,vr)*one_over_r(r)**2 &
                + buffer(PSI,dvrdt)*buffer(PSI,vphi)*one_over_r(r)**2 &
                - buffer(PSI,vphi)*buffer(PSI,vtheta)*one_over_r(r)**2 &
                + DDBUFF(PSI,dvpdtdp)*buffer(PSI,vphi)*csctheta(t)*one_over_r(r)**2 &
                + buffer(PSI,dvpdp)*buffer(PSI,dvpdt)*csctheta(t)*one_over_r(r)**2 &
                - DDBUFF(PSI,dvtdpdp)*buffer(PSI,vphi)*csctheta(t)**2*one_over_r(r)**2 &
                - DDBUFF(PSI,dvtdrdp)*buffer(PSI,vr)*csctheta(t)*one_over_r(r) &
                - DDBUFF(PSI,dvtdtdp)*buffer(PSI,vtheta)*csctheta(t)*one_over_r(r)**2 &
                - buffer(PSI,dvpdp)*buffer(PSI,dvtdp)*csct

`vgv_abs_t` (Fortran-symbol form)

In [18]:
print(fortran_lines(vgv_abs_t, lhs='vgv_abs_t = '))

                vgv_abs_t = -DDBUFF(PSI,dvpdrdr)*buffer(PSI,vr) - buffer(PSI,dvpdr)*buffer(PSI,dvrdr) &
                - DDBUFF(PSI,dvpdrdt)*buffer(PSI,vtheta)*one_over_r(r) &
                - buffer(PSI,dvpdr)*buffer(PSI,vr)*ref%dlnrho(r) - buffer(PSI,dvpdt)*buffer(PSI,dvtdr)*one_over_r(r) &
                - buffer(PSI,dvrdr)*buffer(PSI,vphi)*one_over_r(r) &
                - 2*buffer(PSI,dvpdr)*buffer(PSI,vr)*one_over_r(r) &
                + DDBUFF(PSI,dvrdpdp)*buffer(PSI,vphi)*csctheta(t)**2*one_over_r(r)**2 &
                + DDBUFF(PSI,dvrdrdp)*buffer(PSI,vr)*csctheta(t)*one_over_r(r) &
                + DDBUFF(PSI,dvrdtdp)*buffer(PSI,vtheta)*csctheta(t)*one_over_r(r)**2 &
                + buffer(PSI,dvpdp)*buffer(PSI,dvrdp)*csctheta(t)**2*one_over_r(r)**2 &
                + buffer(PSI,dvrdp)*buffer(PSI,dvrdr)*csctheta(t)*one_over_r(r) &
                + buffer(PSI,dvrdt)*buffer(PSI,dvtdp)*csctheta(t)*one_over_r(r)**2 &
                - DDBUFF(PSI,dvpdrdp)*buffer(PSI,vphi

`vgv_abs_p` (Fortran-symbol form)

In [19]:
print(fortran_lines(vgv_abs_p, lhs='vgv_abs_p = '))

                vgv_abs_p = DDBUFF(PSI,dvtdrdr)*buffer(PSI,vr) + buffer(PSI,dvrdr)*buffer(PSI,dvtdr) &
                + DDBUFF(PSI,dvtdrdt)*buffer(PSI,vtheta)*one_over_r(r) &
                + buffer(PSI,dvrdr)*buffer(PSI,vtheta)*one_over_r(r) &
                + buffer(PSI,dvtdr)*buffer(PSI,dvtdt)*one_over_r(r) + buffer(PSI,dvtdr)*buffer(PSI,vr)*ref%dlnrho(r) &
                - DDBUFF(PSI,dvrdrdt)*buffer(PSI,vr)*one_over_r(r) &
                - DDBUFF(PSI,dvrdtdt)*buffer(PSI,vtheta)*one_over_r(r)**2 &
                - buffer(PSI,dvrdr)*buffer(PSI,dvrdt)*one_over_r(r) &
                - buffer(PSI,dvrdt)*buffer(PSI,dvtdt)*one_over_r(r)**2 &
                + 2*buffer(PSI,dvpdt)*buffer(PSI,vphi)*one_over_r(r)**2 &
                + 2*buffer(PSI,dvtdr)*buffer(PSI,vr)*one_over_r(r) &
                + 2*buffer(PSI,dvtdt)*buffer(PSI,vtheta)*one_over_r(r)**2 &
                + DDBUFF(PSI,dvtdrdp)*buffer(PSI,vphi)*csctheta(t)*one_over_r(r) &
                + buffer(PSI,dvpdr)*buffer(P

## 6. Magnetic (Lorentz) force $L_c\,(\nabla\times B)\times B$ (`j_cross_b`)

Baseline (`Diagnostics_Lorentz_Forces.F90`), with $L_c=$`ref%Lorentz_Coeff`
and $J=\nabla\times B$ (the current density, up to the constant folded into
$L_c$):

$$F = L_c\,(\nabla\times B)\times B,\qquad
F_r = L_c(J_\theta B_\phi - J_\phi B_\theta),\quad
F_\theta = L_c(J_\phi B_r - J_r B_\phi),\quad
F_\phi = L_c(J_r B_\theta - J_\theta B_r)$$

In [20]:

Lc = sp.Symbol('L_c')
Br = sp.Function('B_r')(r, theta, phi)
Bt = sp.Function('B_t')(r, theta, phi)
Bp = sp.Function('B_p')(r, theta, phi)

dbrdr, dbrdt, dbrdp = sp.diff(Br,r), sp.diff(Br,theta), sp.diff(Br,phi)
dbtdr, dbtdt, dbtdp = sp.diff(Bt,r), sp.diff(Bt,theta), sp.diff(Bt,phi)
dbpdr, dbpdt, dbpdp = sp.diff(Bp,r), sp.diff(Bp,theta), sp.diff(Bp,phi)

Jr, Jt, Jp = curl(Br, Bt, Bp)

Fr_j = Lc*(Jt*Bp - Jp*Bt)
Ft_j = Lc*(Jp*Br - Jr*Bp)
Fp_j = Lc*(Jr*Bt - Jt*Br)

Fr_j, Ft_j, Fp_j


⎛    ⎛⎛                                    ∂              ⎞                   
⎜    ⎜⎜                                    ──(Bᵣ(r, θ, φ))⎟                   
⎜    ⎜⎜    ∂                               ∂φ             ⎟               ⎛  ∂
⎜    ⎜⎜- r⋅──(Bₚ(r, θ, φ)) - Bₚ(r, θ, φ) + ───────────────⎟⋅Bₚ(r, θ, φ)   ⎜r⋅─
⎜    ⎜⎝    ∂r                                   sin(θ)    ⎠               ⎝  ∂
⎜L_c⋅⎜───────────────────────────────────────────────────────────────── - ────
⎝    ⎝                                r                                       

                                                           ⎞      ⎛           
                                                           ⎟      ⎜           
                               ∂              ⎞            ⎟      ⎜⎛  ∂       
─(Bₜ(r, θ, φ)) + Bₜ(r, θ, φ) - ──(Bᵣ(r, θ, φ))⎟⋅Bₜ(r, θ, φ)⎟      ⎜⎜r⋅──(Bₜ(r,
r                              ∂θ             ⎠            ⎟      ⎜⎝  ∂r      
───────────────────────────────────────────────────

### True curl (symbolic)

$\nabla\times[L_c(\nabla\times B)\times B]$ involves up to second derivatives
of $B$ (one derivative from the inner curl, one more from the outer curl),
matching the `DDBUFF` second-derivative terms already used in the code for
this diagnostic.

In [21]:

cr_j, ct_j, cp_j = curl(Fr_j, Ft_j, Fp_j)
cr_j, ct_j, cp_j


⎛    ⎛                                                                        
⎜    ⎜                                                                        
⎜    ⎜                                                 ∂                      
⎜    ⎜                 2                 r⋅Bᵣ(r, θ, φ)⋅──(Bₚ(r, θ, φ))   r⋅Bᵣ(
⎜    ⎜                ∂                                ∂r                     
⎜L_c⋅⎜r⋅Bᵣ(r, θ, φ)⋅─────(Bₚ(r, θ, φ)) + ───────────────────────────── - ─────
⎜    ⎜              ∂θ ∂r                            tan(θ)                   
⎜    ⎝                                                                        
⎜─────────────────────────────────────────────────────────────────────────────
⎜                                                                             
⎝                                                                             

                                                                              
            2                                      

### Fortranize

Same procedure as for `v_grad_v`: substitute `Derivative`/`Function` objects
for symbols named exactly like the Fortran buffer accessors, then confirm
the substitution was lossless.

In [22]:

fields_j = {'br': Br, 'bt': Bt, 'bp': Bp}
field_buffer_name_j = {'br': 'br', 'bt': 'btheta', 'bp': 'bphi'}

subs_list_j = []
for fshort, ffunc in fields_j.items():
    for a, b in pairs:
        name = f'DDBUFF(PSI,d{fshort}d{a}d{b})'
        subs_list_j.append((sp.diff(ffunc, varsym[a], varsym[b]), sp.Symbol(name)))
for fshort, ffunc in fields_j.items():
    for a in ['r','t','p']:
        name = f'buffer(PSI,d{fshort}d{a})'
        subs_list_j.append((sp.diff(ffunc, varsym[a]), sp.Symbol(name)))
for fshort, ffunc in fields_j.items():
    name = f'buffer(PSI,{field_buffer_name_j[fshort]})'
    subs_list_j.append((ffunc, sp.Symbol(name)))

LC = sp.Symbol('ref%Lorentz_Coeff')
metric_subs_j = [(oor, OOR), (csc, CSC), (cot, COT), (cos_, COS_),
                  (1/sp.tan(theta), COT), (sp.tan(theta), 1/COT), (Lc, LC)]

def fortranize_j(expr):
    e = expr.subs(subs_list_j)
    e = sp.expand(e)
    e = e.subs(metric_subs_j)
    e = sp.expand(e)
    return e

fr_out_j = fortranize_j(cr_j)
ft_out_j = fortranize_j(ct_j)
fp_out_j = fortranize_j(cp_j)

for name, e in [("curl_j_cross_b_r", fr_out_j), ("curl_j_cross_b_theta", ft_out_j), ("curl_j_cross_b_phi", fp_out_j)]:
    residual_atoms = [a for a in e.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
    print(f"{name}: {'CLEAN (fully expressed in Fortran symbols)' if not residual_atoms else 'LEFTOVER: '+str(residual_atoms)}")


curl_j_cross_b_r: CLEAN (fully expressed in Fortran symbols)
curl_j_cross_b_theta: CLEAN (fully expressed in Fortran symbols)
curl_j_cross_b_phi: CLEAN (fully expressed in Fortran symbols)


### `curl_j_cross_b_r` (Fortran-symbol form)

This is exactly the expression now coded at
`Diagnostics_Curl_Momentum.F90`'s `curl_j_cross_b_r` block.

In [23]:
print(fortran_lines(fr_out_j))

                qty(PSI) = DDBUFF(PSI,dbpdrdt)*buffer(PSI,br)*one_over_r(r)*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbpdtdt)*buffer(PSI,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,bphi)*buffer(PSI,dbrdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,br)*buffer(PSI,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,dbpdr)*buffer(PSI,dbrdt)*one_over_r(r)*ref%Lorentz_Coeff &
                + buffer(PSI,dbpdt)*buffer(PSI,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - buffer(PSI,bphi)*buffer(PSI,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbpdtdp)*buffer(PSI,bphi)*csctheta(t)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,bphi)*buffer(PSI,br)*cottheta(t)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,bphi)*buffer(PSI,dbtdt)*cottheta(t)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,br)*buffer(PSI,dbpdr)*cotthe

### `curl_j_cross_b_theta` (Fortran-symbol form)

In [24]:
print(fortran_lines(ft_out_j))

                qty(PSI) = -DDBUFF(PSI,dbpdrdr)*buffer(PSI,br)*ref%Lorentz_Coeff &
                - buffer(PSI,dbpdr)*buffer(PSI,dbrdr)*ref%Lorentz_Coeff &
                - DDBUFF(PSI,dbpdrdt)*buffer(PSI,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                - buffer(PSI,bphi)*buffer(PSI,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - buffer(PSI,dbpdt)*buffer(PSI,dbtdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - 2*buffer(PSI,br)*buffer(PSI,dbpdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbrdpdp)*buffer(PSI,bphi)*csctheta(t)**2*one_over_r(r)**2*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbrdrdp)*buffer(PSI,br)*csctheta(t)*one_over_r(r)*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbrdtdp)*buffer(PSI,btheta)*csctheta(t)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,dbpdp)*buffer(PSI,dbrdp)*csctheta(t)**2*one_over_r(r)**2*ref%Lorentz_Coeff &
                + buffer(PSI,dbrdp)*buffer(PSI,dbrdr)*csctheta(t)*one_ov

### `curl_j_cross_b_phi` (Fortran-symbol form)

In [25]:
print(fortran_lines(fp_out_j))

                qty(PSI) = DDBUFF(PSI,dbtdrdr)*buffer(PSI,br)*ref%Lorentz_Coeff &
                + buffer(PSI,dbrdr)*buffer(PSI,dbtdr)*ref%Lorentz_Coeff &
                + DDBUFF(PSI,dbtdrdt)*buffer(PSI,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                + buffer(PSI,btheta)*buffer(PSI,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + buffer(PSI,dbtdr)*buffer(PSI,dbtdt)*one_over_r(r)*ref%Lorentz_Coeff &
                - DDBUFF(PSI,dbrdrdt)*buffer(PSI,br)*one_over_r(r)*ref%Lorentz_Coeff &
                - DDBUFF(PSI,dbrdtdt)*buffer(PSI,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - buffer(PSI,dbrdr)*buffer(PSI,dbrdt)*one_over_r(r)*ref%Lorentz_Coeff &
                - buffer(PSI,dbrdt)*buffer(PSI,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + 2*buffer(PSI,bphi)*buffer(PSI,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + 2*buffer(PSI,br)*buffer(PSI,dbtdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + 2*buffer(PSI,btheta

### `curl_j_cross_b_abs`

Coded as $L_c\sqrt{\hat r^2+\hat\theta^2+\hat\phi^2}$ where $\hat r,\hat\theta,\hat\phi$
are the `curl_j_cross_b_r/theta/phi` formulas above with the common factor
`ref%Lorentz_Coeff` divided out. Verify that factoring is exact before
trusting the `jxb_abs_r/t/p` local temporaries in the code.

In [26]:

jxb_abs_r = sp.simplify(sp.expand(fr_out_j)/LC)
jxb_abs_t = sp.simplify(sp.expand(ft_out_j)/LC)
jxb_abs_p = sp.simplify(sp.expand(fp_out_j)/LC)

for name, full, bracket in [("r", fr_out_j, jxb_abs_r), ("theta", ft_out_j, jxb_abs_t), ("phi", fp_out_j, jxb_abs_p)]:
    residual = sp.simplify(full - LC*bracket)
    print(f"{name}: Lorentz coefficient factors out cleanly = {residual == 0}")


r: Lorentz coefficient factors out cleanly = True
theta: Lorentz coefficient factors out cleanly = True
phi: Lorentz coefficient factors out cleanly = True


`jxb_abs_r` (Fortran-symbol form, `ref%Lorentz_Coeff` divided out)

In [27]:
print(fortran_lines(jxb_abs_r, lhs='jxb_abs_r = '))

                jxb_abs_r = DDBUFF(PSI,dbpdrdt)*buffer(PSI,br)*one_over_r(r) &
                + DDBUFF(PSI,dbpdtdt)*buffer(PSI,btheta)*one_over_r(r)**2 &
                + buffer(PSI,bphi)*buffer(PSI,dbrdt)*one_over_r(r)**2 &
                + buffer(PSI,br)*buffer(PSI,dbpdt)*one_over_r(r)**2 &
                + buffer(PSI,dbpdr)*buffer(PSI,dbrdt)*one_over_r(r) &
                + buffer(PSI,dbpdt)*buffer(PSI,dbtdt)*one_over_r(r)**2 &
                - buffer(PSI,bphi)*buffer(PSI,btheta)*one_over_r(r)**2 &
                + DDBUFF(PSI,dbpdtdp)*buffer(PSI,bphi)*csctheta(t)*one_over_r(r)**2 &
                + buffer(PSI,bphi)*buffer(PSI,br)*cottheta(t)*one_over_r(r)**2 &
                + buffer(PSI,bphi)*buffer(PSI,dbtdt)*cottheta(t)*one_over_r(r)**2 &
                + buffer(PSI,br)*buffer(PSI,dbpdr)*cottheta(t)*one_over_r(r) &
                + buffer(PSI,dbpdp)*buffer(PSI,dbpdt)*csctheta(t)*one_over_r(r)**2 &
                - DDBUFF(PSI,dbtdpdp)*buffer(PSI,bphi)*csctheta(t)**2*on

`jxb_abs_t` (Fortran-symbol form)

In [28]:
print(fortran_lines(jxb_abs_t, lhs='jxb_abs_t = '))

                jxb_abs_t = -DDBUFF(PSI,dbpdrdr)*buffer(PSI,br) - buffer(PSI,dbpdr)*buffer(PSI,dbrdr) &
                - DDBUFF(PSI,dbpdrdt)*buffer(PSI,btheta)*one_over_r(r) &
                - buffer(PSI,bphi)*buffer(PSI,dbrdr)*one_over_r(r) &
                - buffer(PSI,dbpdt)*buffer(PSI,dbtdr)*one_over_r(r) &
                - 2*buffer(PSI,br)*buffer(PSI,dbpdr)*one_over_r(r) &
                + DDBUFF(PSI,dbrdpdp)*buffer(PSI,bphi)*csctheta(t)**2*one_over_r(r)**2 &
                + DDBUFF(PSI,dbrdrdp)*buffer(PSI,br)*csctheta(t)*one_over_r(r) &
                + DDBUFF(PSI,dbrdtdp)*buffer(PSI,btheta)*csctheta(t)*one_over_r(r)**2 &
                + buffer(PSI,dbpdp)*buffer(PSI,dbrdp)*csctheta(t)**2*one_over_r(r)**2 &
                + buffer(PSI,dbrdp)*buffer(PSI,dbrdr)*csctheta(t)*one_over_r(r) &
                + buffer(PSI,dbrdt)*buffer(PSI,dbtdp)*csctheta(t)*one_over_r(r)**2 &
                - DDBUFF(PSI,dbpdrdp)*buffer(PSI,bphi)*csctheta(t)*one_over_r(r) &
                - b

`jxb_abs_p` (Fortran-symbol form)

In [29]:
print(fortran_lines(jxb_abs_p, lhs='jxb_abs_p = '))

                jxb_abs_p = DDBUFF(PSI,dbtdrdr)*buffer(PSI,br) + buffer(PSI,dbrdr)*buffer(PSI,dbtdr) &
                + DDBUFF(PSI,dbtdrdt)*buffer(PSI,btheta)*one_over_r(r) &
                + buffer(PSI,btheta)*buffer(PSI,dbrdr)*one_over_r(r) &
                + buffer(PSI,dbtdr)*buffer(PSI,dbtdt)*one_over_r(r) &
                - DDBUFF(PSI,dbrdrdt)*buffer(PSI,br)*one_over_r(r) &
                - DDBUFF(PSI,dbrdtdt)*buffer(PSI,btheta)*one_over_r(r)**2 &
                - buffer(PSI,dbrdr)*buffer(PSI,dbrdt)*one_over_r(r) &
                - buffer(PSI,dbrdt)*buffer(PSI,dbtdt)*one_over_r(r)**2 &
                + 2*buffer(PSI,bphi)*buffer(PSI,dbpdt)*one_over_r(r)**2 &
                + 2*buffer(PSI,br)*buffer(PSI,dbtdr)*one_over_r(r) &
                + 2*buffer(PSI,btheta)*buffer(PSI,dbtdt)*one_over_r(r)**2 &
                + DDBUFF(PSI,dbtdrdp)*buffer(PSI,bphi)*csctheta(t)*one_over_r(r) &
                + buffer(PSI,dbpdr)*buffer(PSI,dbtdp)*csctheta(t)*one_over_r(r) &
           

## 7. Reuse: `curl_vp_grad_vp_*` = `curl_v_grad_v_*` with `v` &#8594; `v'`

`v_grad_v` is $\rho\,(v\cdot\nabla)v$ — a field dotted-and-differentiated
against *itself*. `vp_grad_vp` is $\rho\,(v'\cdot\nabla)v'$: the exact same
operator, with the fluctuating (primed) velocity playing both roles instead
of the full velocity. Since $v'$ has the same functional form as $v$ (a
generic function of $r,\theta,\phi$ — Rayleigh's `fbuffer` is fully
$\phi$-dependent, only its $\phi$-average has been removed), no new curl
needs to be derived: the verified `fr_out`/`ft_out`/`fp_out` expressions
above transform directly by renaming every `buffer(PSI,...)` &#8594;
`fbuffer(PSI,...)` and `DDBUFF(PSI,...)` &#8594; `d2_fbuffer(PSI,...)`
(`d2_fbuffer` is the fluctuating-field counterpart of `DDBUFF`'s
`d2buffer%p3a`, populated by the same second-derivative pass).

In [30]:

def rename_to_fluctuating(expr):
    subs = {}
    for a in expr.atoms(sp.Symbol):
        name = a.name
        if name.startswith('buffer(PSI,'):
            subs[a] = sp.Symbol('f' + name)
        elif name.startswith('DDBUFF(PSI,'):
            subs[a] = sp.Symbol('d2_fbuffer(PSI,' + name[len('DDBUFF(PSI,'):])
    return expr.subs(subs)

fr_out_pp = rename_to_fluctuating(fr_out)
ft_out_pp = rename_to_fluctuating(ft_out)
fp_out_pp = rename_to_fluctuating(fp_out)


`curl_vp_grad_vp_r` (Fortran-symbol form)

In [31]:
print(fortran_lines(fr_out_pp))

                qty(PSI) = d2_fbuffer(PSI,dvpdrdt)*fbuffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                + d2_fbuffer(PSI,dvpdtdt)*fbuffer(PSI,vtheta)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,dvpdr)*fbuffer(PSI,dvrdt)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvpdt)*fbuffer(PSI,dvtdt)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,dvpdt)*fbuffer(PSI,vr)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,dvrdt)*fbuffer(PSI,vphi)*one_over_r(r)**2*ref%density(r) &
                - fbuffer(PSI,vphi)*fbuffer(PSI,vtheta)*one_over_r(r)**2*ref%density(r) &
                + csctheta(t)*d2_fbuffer(PSI,dvpdtdp)*fbuffer(PSI,vphi)*one_over_r(r)**2*ref%density(r) &
                + csctheta(t)*fbuffer(PSI,dvpdp)*fbuffer(PSI,dvpdt)*one_over_r(r)**2*ref%density(r) &
                - csctheta(t)*d2_fbuffer(PSI,dvtdrdp)*fbuffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                - csctheta(t)*d2_fbuffer(PSI,dvtdtdp)*

`curl_vp_grad_vp_theta` (Fortran-symbol form)

In [32]:
print(fortran_lines(ft_out_pp))

                qty(PSI) = -d2_fbuffer(PSI,dvpdrdr)*fbuffer(PSI,vr)*ref%density(r) &
                - fbuffer(PSI,dvpdr)*fbuffer(PSI,dvrdr)*ref%density(r) &
                - d2_fbuffer(PSI,dvpdrdt)*fbuffer(PSI,vtheta)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,dvpdr)*fbuffer(PSI,vr)*ref%density(r)*ref%dlnrho(r) &
                - fbuffer(PSI,dvpdt)*fbuffer(PSI,dvtdr)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,dvrdr)*fbuffer(PSI,vphi)*one_over_r(r)*ref%density(r) &
                - 2*fbuffer(PSI,dvpdr)*fbuffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                + csctheta(t)*d2_fbuffer(PSI,dvrdrdp)*fbuffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                + csctheta(t)*d2_fbuffer(PSI,dvrdtdp)*fbuffer(PSI,vtheta)*one_over_r(r)**2*ref%density(r) &
                + csctheta(t)*fbuffer(PSI,dvrdp)*fbuffer(PSI,dvrdr)*one_over_r(r)*ref%density(r) &
                + csctheta(t)*fbuffer(PSI,dvrdt)*fbuffer(PSI,dvtdp)*one_over_r(r)**2*ref%density(r

`curl_vp_grad_vp_phi` (Fortran-symbol form)

In [33]:
print(fortran_lines(fp_out_pp))

                qty(PSI) = d2_fbuffer(PSI,dvtdrdr)*fbuffer(PSI,vr)*ref%density(r) &
                + fbuffer(PSI,dvrdr)*fbuffer(PSI,dvtdr)*ref%density(r) &
                + d2_fbuffer(PSI,dvtdrdt)*fbuffer(PSI,vtheta)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvrdr)*fbuffer(PSI,vtheta)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvtdr)*fbuffer(PSI,dvtdt)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvtdr)*fbuffer(PSI,vr)*ref%density(r)*ref%dlnrho(r) &
                - d2_fbuffer(PSI,dvrdrdt)*fbuffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                - d2_fbuffer(PSI,dvrdtdt)*fbuffer(PSI,vtheta)*one_over_r(r)**2*ref%density(r) &
                - fbuffer(PSI,dvrdr)*fbuffer(PSI,dvrdt)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,dvrdt)*fbuffer(PSI,dvtdt)*one_over_r(r)**2*ref%density(r) &
                + 2*fbuffer(PSI,dvpdt)*fbuffer(PSI,vphi)*one_over_r(r)**2*ref%density(r) &
                + 2*fbuffer(PSI,dvtdr)

## 8. Reuse: `curl_jp_cross_bp_*` = `curl_j_cross_b_*` with `B` &#8594; `B'`

Same principle as above, applied to the magnetic force: `jp_cross_bp` is
$L_c(\nabla\times B')\times B'$, structurally identical to the full
`j_cross_b` = $L_c(\nabla\times B)\times B$ with $B'$ (Rayleigh's `fbuffer`
applied to the magnetic field components) playing both roles. The verified
`fr_out_j`/`ft_out_j`/`fp_out_j` expressions transform the same way.

In [34]:

fr_out_j_pp = rename_to_fluctuating(fr_out_j)
ft_out_j_pp = rename_to_fluctuating(ft_out_j)
fp_out_j_pp = rename_to_fluctuating(fp_out_j)


`curl_jp_cross_bp_r` (Fortran-symbol form)

In [35]:
print(fortran_lines(fr_out_j_pp))

                qty(PSI) = d2_fbuffer(PSI,dbpdrdt)*fbuffer(PSI,br)*one_over_r(r)*ref%Lorentz_Coeff &
                + d2_fbuffer(PSI,dbpdtdt)*fbuffer(PSI,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,bphi)*fbuffer(PSI,dbrdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,br)*fbuffer(PSI,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbpdr)*fbuffer(PSI,dbrdt)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbpdt)*fbuffer(PSI,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - fbuffer(PSI,bphi)*fbuffer(PSI,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + cottheta(t)*fbuffer(PSI,bphi)*fbuffer(PSI,br)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + cottheta(t)*fbuffer(PSI,bphi)*fbuffer(PSI,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + cottheta(t)*fbuffer(PSI,br)*fbuffer(PSI,dbpdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + csctheta(t)*d2_fbuff

`curl_jp_cross_bp_theta` (Fortran-symbol form)

In [36]:
print(fortran_lines(ft_out_j_pp))

                qty(PSI) = -d2_fbuffer(PSI,dbpdrdr)*fbuffer(PSI,br)*ref%Lorentz_Coeff &
                - fbuffer(PSI,dbpdr)*fbuffer(PSI,dbrdr)*ref%Lorentz_Coeff &
                - d2_fbuffer(PSI,dbpdrdt)*fbuffer(PSI,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                - fbuffer(PSI,bphi)*fbuffer(PSI,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - fbuffer(PSI,dbpdt)*fbuffer(PSI,dbtdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - 2*fbuffer(PSI,br)*fbuffer(PSI,dbpdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + csctheta(t)*d2_fbuffer(PSI,dbrdrdp)*fbuffer(PSI,br)*one_over_r(r)*ref%Lorentz_Coeff &
                + csctheta(t)*d2_fbuffer(PSI,dbrdtdp)*fbuffer(PSI,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + csctheta(t)*fbuffer(PSI,dbrdp)*fbuffer(PSI,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + csctheta(t)*fbuffer(PSI,dbrdt)*fbuffer(PSI,dbtdp)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + csctheta(t)**2*d2_fbuffer(PSI,db

`curl_jp_cross_bp_phi` (Fortran-symbol form)

In [37]:
print(fortran_lines(fp_out_j_pp))

                qty(PSI) = d2_fbuffer(PSI,dbtdrdr)*fbuffer(PSI,br)*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbrdr)*fbuffer(PSI,dbtdr)*ref%Lorentz_Coeff &
                + d2_fbuffer(PSI,dbtdrdt)*fbuffer(PSI,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,btheta)*fbuffer(PSI,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbtdr)*fbuffer(PSI,dbtdt)*one_over_r(r)*ref%Lorentz_Coeff &
                - d2_fbuffer(PSI,dbrdrdt)*fbuffer(PSI,br)*one_over_r(r)*ref%Lorentz_Coeff &
                - d2_fbuffer(PSI,dbrdtdt)*fbuffer(PSI,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - fbuffer(PSI,dbrdr)*fbuffer(PSI,dbrdt)*one_over_r(r)*ref%Lorentz_Coeff &
                - fbuffer(PSI,dbrdt)*fbuffer(PSI,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + 2*fbuffer(PSI,bphi)*fbuffer(PSI,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + 2*fbuffer(PSI,br)*fbuffer(PSI,dbtdr)*one_over_r(r)*ref%Lorentz_Coeff &
   

## 9. Reuse: `curl_viscous_pforce/mforce_*` = `curl_viscous_force_*`

Unlike the advection and Lorentz cases above, `curl_viscous_force_*` (Section
4) was never tied to a specific physical vector field in its derivation — it
is the generic spherical curl of *whatever* vector field
`Grad_Viscous_Force` differentiates. Rayleigh already computes derivatives
of the fluctuating and mean viscous force fields into the same `VFDBUFF`
buffer at different offsets (`vfp_r/t/p`, `vfm_r/t/p` and their
`dvfp_*_d*`/`dvfm_*_d*` derivative indices) — the code just never had a
`qty(PSI) = ...` block reading them out for `curl_viscous_pforce_r/theta/phi`
and `curl_viscous_mforce_r/theta/phi`. No new symbolic derivation is
needed; Section 4's generic-vector-curl formula is reused verbatim with the
buffer offsets swapped in.

## 10. `curl_vm_grad_vm_*`: axisymmetric self-advection

$\overline v$ (Rayleigh's `m0_values`) is the $\phi$-average of $v$, so it
is a function of $r,\theta$ only — `m0_values` is a 2D `(r,theta)` array
with no $\phi$ index at all. This is a fresh derivation (not a substitution
on Section 5's result): $v_r,v_\theta,v_\phi$ are redefined below as
functions of $r,\theta$ only, so every $\partial_\phi$ term in the
$(v\cdot\nabla)v$ and curl formulas vanishes identically by construction,
rather than by cancellation.

In [38]:

vmr = sp.Function('v_r')(r, theta)
vmt = sp.Function('v_t')(r, theta)
vmp = sp.Function('v_p')(r, theta)

dvmrdr, dvmrdt = sp.diff(vmr,r), sp.diff(vmr,theta)
dvmtdr, dvmtdt = sp.diff(vmt,r), sp.diff(vmt,theta)
dvmpdr, dvmpdt = sp.diff(vmp,r), sp.diff(vmp,theta)

vgv_mr = vmr*dvmrdr + vmt*oor*dvmrdt - (vmt**2+vmp**2)*oor
vgv_mt = vmr*dvmtdr + vmt*oor*dvmtdt + vmr*vmt*oor - vmp**2*cot*oor
vgv_mp = vmr*dvmpdr + vmt*oor*dvmpdt + vmr*vmp*oor + vmt*vmp*cot*oor

Fr_m = rho*vgv_mr
Ft_m = rho*vgv_mt
Fp_m = rho*vgv_mp

cr_m, ct_m, cp_m = curl(Fr_m, Ft_m, Fp_m)
cr_m, ct_m, cp_m


⎛⎛                                        ∂                                   
⎜⎜              2              r⋅vᵣ(r, θ)⋅──(vₚ(r, θ))                        
⎜⎜             ∂                          ∂r               ∂            ∂     
⎜⎜r⋅vᵣ(r, θ)⋅─────(vₚ(r, θ)) + ─────────────────────── + r⋅──(vₚ(r, θ))⋅──(vᵣ(
⎜⎜           ∂θ ∂r                      tan(θ)             ∂r           ∂θ    
⎜⎝                                                                            
⎜─────────────────────────────────────────────────────────────────────────────
⎜                                                                             
⎝                                                                             

                                                                              
                                                                         vₚ(r,
         vₚ(r, θ)⋅vᵣ(r, θ)                                ∂                   
r, θ)) + ───────────────── - vₚ(r, θ)⋅vₜ(r, θ) + vₚ

### Fortranize to `m0_values(PSI2,...)`/`d2_m0(PSI2,...)`

`d2_m0` is the axisymmetric-mean counterpart of `DDBUFF`'s `d2buffer%p3a`
(populated by the same second-derivative pass, sharing the same field-index
convention). Only `r`/`theta` derivative pairs are needed here — no `dp`,
`drdp`, `dtdp`, or `dpdp` index ever appears, since `m0_values` genuinely
has no `phi` argument to differentiate against.

In [39]:

fields_m = {'vr': vmr, 'vt': vmt, 'vp': vmp}
field_buffer_name_m = {'vr': 'vr', 'vt': 'vtheta', 'vp': 'vphi'}
varsym_m = {'r': r, 't': theta}

subs_list_m = []
pairs_m = [('r','r'), ('r','t'), ('t','t')]
for fshort, ffunc in fields_m.items():
    for a, b in pairs_m:
        name = f'd2_m0(PSI2,d{fshort}d{a}d{b})'
        subs_list_m.append((sp.diff(ffunc, varsym_m[a], varsym_m[b]), sp.Symbol(name)))
for fshort, ffunc in fields_m.items():
    for a in ['r','t']:
        name = f'm0_values(PSI2,d{fshort}d{a})'
        subs_list_m.append((sp.diff(ffunc, varsym_m[a]), sp.Symbol(name)))
for fshort, ffunc in fields_m.items():
    name = f'm0_values(PSI2,{field_buffer_name_m[fshort]})'
    subs_list_m.append((ffunc, sp.Symbol(name)))

subs_list_m = [(sp.diff(rho, r), DLNRHO*RHO)] + subs_list_m + [(rho, RHO)]

def fortranize_m(expr):
    e = expr.subs(subs_list_m)
    e = sp.expand(e)
    e = e.subs(metric_subs)
    e = sp.expand(e)
    return e

fr_out_m = fortranize_m(cr_m)
ft_out_m = fortranize_m(ct_m)
fp_out_m = fortranize_m(cp_m)

for name, e in [("curl_vm_grad_vm_r", fr_out_m), ("curl_vm_grad_vm_theta", ft_out_m), ("curl_vm_grad_vm_phi", fp_out_m)]:
    residual_atoms = [a for a in e.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
    print(f"{name}: {'CLEAN (fully expressed in Fortran symbols)' if not residual_atoms else 'LEFTOVER: '+str(residual_atoms)}")


curl_vm_grad_vm_r: CLEAN (fully expressed in Fortran symbols)
curl_vm_grad_vm_theta: CLEAN (fully expressed in Fortran symbols)
curl_vm_grad_vm_phi: CLEAN (fully expressed in Fortran symbols)


### Cross-check against Section 5

As a second, fully independent confirmation (working symbolically, before
either side is Fortranized): take `cr, ct, cp` — the *general* curl(v.grad v)
result from Section 5, still in terms of the 3-argument $v(r,\theta,\phi)$ —
and literally zero out every `Derivative` with respect to `phi` wherever it
appears (this is exactly what axisymmetry means, applied directly to the
already-verified general formula, rather than re-deriving anything). Then
relabel the now-phi-derivative-free 3-argument functions/derivatives as the
corresponding 2-argument $\overline v(r,\theta)$ objects used in this
section, and confirm the result is identical to `cr_m, ct_m, cp_m` above.
Naively filtering the *Fortranized* text for terms containing `csctheta(t)`
(an earlier attempt at this check) is unreliable: some legitimate,
non-`phi`-derivative curvature terms are algebraically $\cot\theta$ but
happen to print as the literal product `costheta(t)*csctheta(t)` rather
than the single symbol `cottheta(t)`, so a text-level filter both drops
terms it shouldn't and is blind to why a term is actually `phi`-derivative
sourced.

In [40]:

def zero_phi_derivatives(expr):
    subs = {}
    for d in expr.atoms(sp.Derivative):
        if phi in d.variables:
            subs[d] = 0
    return expr.subs(subs)

def relabel_to_axisymmetric(expr, pairs_3to2):
    subs = {}
    for f3, f2 in pairs_3to2:
        subs[f3] = f2
        for order in [(r,), (theta,), (r,r), (r,theta), (theta,theta)]:
            subs[sp.Derivative(f3, *order)] = sp.Derivative(f2, *order)
    return expr.subs(subs)

pairs_v_3to2 = [(vr, vmr), (vt, vmt), (vp, vmp)]

for name, full, mean in [("r", cr_a, cr_m), ("theta", ct_a, ct_m), ("phi", cp_a, cp_m)]:
    check = relabel_to_axisymmetric(zero_phi_derivatives(full), pairs_v_3to2)
    print(f"{name}: cross-check against Section 5's general curl(v.grad v), "
          f"phi-derivatives zeroed, matches = {sp.simplify(check - mean) == 0}")


r: cross-check against Section 5's general curl(v.grad v), phi-derivatives zeroed, matches = True
theta: cross-check against Section 5's general curl(v.grad v), phi-derivatives zeroed, matches = True
phi: cross-check against Section 5's general curl(v.grad v), phi-derivatives zeroed, matches = True


`curl_vm_grad_vm_r` (Fortran-symbol form)

In [41]:
print(fortran_lines(fr_out_m))

                qty(PSI) = d2_m0(PSI2,dvpdrdt)*m0_values(PSI2,vr)*one_over_r(r)*ref%density(r) &
                + d2_m0(PSI2,dvpdtdt)*m0_values(PSI2,vtheta)*one_over_r(r)**2*ref%density(r) &
                + m0_values(PSI2,dvpdr)*m0_values(PSI2,dvrdt)*one_over_r(r)*ref%density(r) &
                + m0_values(PSI2,dvpdt)*m0_values(PSI2,dvtdt)*one_over_r(r)**2*ref%density(r) &
                + m0_values(PSI2,dvpdt)*m0_values(PSI2,vr)*one_over_r(r)**2*ref%density(r) &
                + m0_values(PSI2,dvrdt)*m0_values(PSI2,vphi)*one_over_r(r)**2*ref%density(r) &
                - m0_values(PSI2,vphi)*m0_values(PSI2,vtheta)*one_over_r(r)**2*ref%density(r) &
                + cottheta(t)*m0_values(PSI2,dvpdr)*m0_values(PSI2,vr)*one_over_r(r)*ref%density(r) &
                + cottheta(t)*m0_values(PSI2,dvtdt)*m0_values(PSI2,vphi)*one_over_r(r)**2*ref%density(r) &
                + cottheta(t)*m0_values(PSI2,vphi)*m0_values(PSI2,vr)*one_over_r(r)**2*ref%density(r) &
                + 2*co

`curl_vm_grad_vm_theta` (Fortran-symbol form)

In [42]:
print(fortran_lines(ft_out_m))

                qty(PSI) = -d2_m0(PSI2,dvpdrdr)*m0_values(PSI2,vr)*ref%density(r) &
                - m0_values(PSI2,dvpdr)*m0_values(PSI2,dvrdr)*ref%density(r) &
                - d2_m0(PSI2,dvpdrdt)*m0_values(PSI2,vtheta)*one_over_r(r)*ref%density(r) &
                - m0_values(PSI2,dvpdr)*m0_values(PSI2,vr)*ref%density(r)*ref%dlnrho(r) &
                - m0_values(PSI2,dvpdt)*m0_values(PSI2,dvtdr)*one_over_r(r)*ref%density(r) &
                - m0_values(PSI2,dvrdr)*m0_values(PSI2,vphi)*one_over_r(r)*ref%density(r) &
                - 2*m0_values(PSI2,dvpdr)*m0_values(PSI2,vr)*one_over_r(r)*ref%density(r) &
                - cottheta(t)*m0_values(PSI2,dvpdr)*m0_values(PSI2,vtheta)*one_over_r(r)*ref%density(r) &
                - cottheta(t)*m0_values(PSI2,dvtdr)*m0_values(PSI2,vphi)*one_over_r(r)*ref%density(r) &
                - m0_values(PSI2,dvpdt)*m0_values(PSI2,vtheta)*one_over_r(r)*ref%density(r)*ref%dlnrho(r) &
                - m0_values(PSI2,vphi)*m0_values(PSI2,vr)*on

`curl_vm_grad_vm_phi` (Fortran-symbol form)

In [43]:
print(fortran_lines(fp_out_m))

                qty(PSI) = d2_m0(PSI2,dvtdrdr)*m0_values(PSI2,vr)*ref%density(r) &
                + m0_values(PSI2,dvrdr)*m0_values(PSI2,dvtdr)*ref%density(r) &
                + d2_m0(PSI2,dvtdrdt)*m0_values(PSI2,vtheta)*one_over_r(r)*ref%density(r) &
                + m0_values(PSI2,dvrdr)*m0_values(PSI2,vtheta)*one_over_r(r)*ref%density(r) &
                + m0_values(PSI2,dvtdr)*m0_values(PSI2,dvtdt)*one_over_r(r)*ref%density(r) &
                + m0_values(PSI2,dvtdr)*m0_values(PSI2,vr)*ref%density(r)*ref%dlnrho(r) &
                - d2_m0(PSI2,dvrdrdt)*m0_values(PSI2,vr)*one_over_r(r)*ref%density(r) &
                - d2_m0(PSI2,dvrdtdt)*m0_values(PSI2,vtheta)*one_over_r(r)**2*ref%density(r) &
                - m0_values(PSI2,dvrdr)*m0_values(PSI2,dvrdt)*one_over_r(r)*ref%density(r) &
                - m0_values(PSI2,dvrdt)*m0_values(PSI2,dvtdt)*one_over_r(r)**2*ref%density(r) &
                + 2*m0_values(PSI2,dvpdt)*m0_values(PSI2,vphi)*one_over_r(r)**2*ref%density(r) &


## 11. `curl_jm_cross_bm_*`: axisymmetric self-Lorentz-force

Same principle as Section 10, applied to $L_c(\nabla\times\overline B)\times
\overline B$: $B_r,B_\theta,B_\phi$ are redefined as functions of
$r,\theta$ only.

In [44]:

Bmr = sp.Function('B_r')(r, theta)
Bmt = sp.Function('B_t')(r, theta)
Bmp = sp.Function('B_p')(r, theta)

cBmr, cBmt, cBmp = curl(Bmr, Bmt, Bmp)   # curl of a phi-independent field

Fr_jm = Lc*(cBmt*Bmp - cBmp*Bmt)
Ft_jm = Lc*(cBmp*Bmr - cBmr*Bmp)
Fp_jm = Lc*(cBmr*Bmt - cBmt*Bmr)

cr_jm, ct_jm, cp_jm = curl(Fr_jm, Ft_jm, Fp_jm)
cr_jm, ct_jm, cp_jm


⎛    ⎛                                        ∂                               
⎜    ⎜              2              r⋅Bᵣ(r, θ)⋅──(Bₚ(r, θ))                    
⎜    ⎜             ∂                          ∂r               ∂            ∂ 
⎜L_c⋅⎜r⋅Bᵣ(r, θ)⋅─────(Bₚ(r, θ)) + ─────────────────────── + r⋅──(Bₚ(r, θ))⋅──
⎜    ⎜           ∂θ ∂r                      tan(θ)             ∂r           ∂θ
⎜    ⎝                                                                        
⎜─────────────────────────────────────────────────────────────────────────────
⎜                                                                             
⎝                                                                             

                                                                              
                                                                             B
             Bₚ(r, θ)⋅Bᵣ(r, θ)                                ∂               
(Bᵣ(r, θ)) + ───────────────── - Bₚ(r, θ)⋅Bₜ(r, θ) 

### Fortranize to `m0_values(PSI2,...)`/`d2_m0(PSI2,...)`

In [45]:

fields_jm = {'br': Bmr, 'bt': Bmt, 'bp': Bmp}
field_buffer_name_jm = {'br': 'br', 'bt': 'btheta', 'bp': 'bphi'}

subs_list_jm = []
for fshort, ffunc in fields_jm.items():
    for a, b in pairs_m:
        name = f'd2_m0(PSI2,d{fshort}d{a}d{b})'
        subs_list_jm.append((sp.diff(ffunc, varsym_m[a], varsym_m[b]), sp.Symbol(name)))
for fshort, ffunc in fields_jm.items():
    for a in ['r','t']:
        name = f'm0_values(PSI2,d{fshort}d{a})'
        subs_list_jm.append((sp.diff(ffunc, varsym_m[a]), sp.Symbol(name)))
for fshort, ffunc in fields_jm.items():
    name = f'm0_values(PSI2,{field_buffer_name_jm[fshort]})'
    subs_list_jm.append((ffunc, sp.Symbol(name)))

metric_subs_jm = [(oor, OOR), (csc, CSC), (cot, COT), (cos_, COS_),
                   (1/sp.tan(theta), COT), (sp.tan(theta), 1/COT), (Lc, LC)]

def fortranize_jm(expr):
    e = expr.subs(subs_list_jm)
    e = sp.expand(e)
    e = e.subs(metric_subs_jm)
    e = sp.expand(e)
    return e

fr_out_jm = fortranize_jm(cr_jm)
ft_out_jm = fortranize_jm(ct_jm)
fp_out_jm = fortranize_jm(cp_jm)

for name, e in [("curl_jm_cross_bm_r", fr_out_jm), ("curl_jm_cross_bm_theta", ft_out_jm), ("curl_jm_cross_bm_phi", fp_out_jm)]:
    residual_atoms = [a for a in e.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
    print(f"{name}: {'CLEAN (fully expressed in Fortran symbols)' if not residual_atoms else 'LEFTOVER: '+str(residual_atoms)}")


curl_jm_cross_bm_r: CLEAN (fully expressed in Fortran symbols)
curl_jm_cross_bm_theta: CLEAN (fully expressed in Fortran symbols)
curl_jm_cross_bm_phi: CLEAN (fully expressed in Fortran symbols)


### Cross-check against Section 6

Same rigorous, symbolic-level style of cross-check as Section 10: zero every
`phi`-derivative directly in Section 6's general `cr_j, ct_j, cp_j`, relabel
the 3-argument $B(r,\theta,\phi)$ objects as the 2-argument
$\overline B(r,\theta)$ ones used here, and confirm the result is identical
to `cr_jm, ct_jm, cp_jm` above.

In [46]:

pairs_B_3to2 = [(Br, Bmr), (Bt, Bmt), (Bp, Bmp)]

for name, full, mean in [("r", cr_j, cr_jm), ("theta", ct_j, ct_jm), ("phi", cp_j, cp_jm)]:
    check = relabel_to_axisymmetric(zero_phi_derivatives(full), pairs_B_3to2)
    print(f"{name}: cross-check against Section 6's general curl(Lc*(curl B) x B), "
          f"phi-derivatives zeroed, matches = {sp.simplify(check - mean) == 0}")


r: cross-check against Section 6's general curl(Lc*(curl B) x B), phi-derivatives zeroed, matches = True
theta: cross-check against Section 6's general curl(Lc*(curl B) x B), phi-derivatives zeroed, matches = True
phi: cross-check against Section 6's general curl(Lc*(curl B) x B), phi-derivatives zeroed, matches = True


`curl_jm_cross_bm_r` (Fortran-symbol form)

In [47]:
print(fortran_lines(fr_out_jm))

                qty(PSI) = d2_m0(PSI2,dbpdrdt)*m0_values(PSI2,br)*one_over_r(r)*ref%Lorentz_Coeff &
                + d2_m0(PSI2,dbpdtdt)*m0_values(PSI2,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + m0_values(PSI2,bphi)*m0_values(PSI2,dbrdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + m0_values(PSI2,br)*m0_values(PSI2,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + m0_values(PSI2,dbpdr)*m0_values(PSI2,dbrdt)*one_over_r(r)*ref%Lorentz_Coeff &
                + m0_values(PSI2,dbpdt)*m0_values(PSI2,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - m0_values(PSI2,bphi)*m0_values(PSI2,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + cottheta(t)*m0_values(PSI2,bphi)*m0_values(PSI2,br)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + cottheta(t)*m0_values(PSI2,bphi)*m0_values(PSI2,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + cottheta(t)*m0_values(PSI2,br)*m0_values(PSI2,dbpdr)*one_over_r(r)*ref%Lorentz_

`curl_jm_cross_bm_theta` (Fortran-symbol form)

In [48]:
print(fortran_lines(ft_out_jm))

                qty(PSI) = -d2_m0(PSI2,dbpdrdr)*m0_values(PSI2,br)*ref%Lorentz_Coeff &
                - m0_values(PSI2,dbpdr)*m0_values(PSI2,dbrdr)*ref%Lorentz_Coeff &
                - d2_m0(PSI2,dbpdrdt)*m0_values(PSI2,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                - m0_values(PSI2,bphi)*m0_values(PSI2,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - m0_values(PSI2,dbpdt)*m0_values(PSI2,dbtdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - 2*m0_values(PSI2,br)*m0_values(PSI2,dbpdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - cottheta(t)*m0_values(PSI2,bphi)*m0_values(PSI2,dbtdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - cottheta(t)*m0_values(PSI2,btheta)*m0_values(PSI2,dbpdr)*one_over_r(r)*ref%Lorentz_Coeff


`curl_jm_cross_bm_phi` (Fortran-symbol form)

In [49]:
print(fortran_lines(fp_out_jm))

                qty(PSI) = d2_m0(PSI2,dbtdrdr)*m0_values(PSI2,br)*ref%Lorentz_Coeff &
                + m0_values(PSI2,dbrdr)*m0_values(PSI2,dbtdr)*ref%Lorentz_Coeff &
                + d2_m0(PSI2,dbtdrdt)*m0_values(PSI2,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                + m0_values(PSI2,btheta)*m0_values(PSI2,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + m0_values(PSI2,dbtdr)*m0_values(PSI2,dbtdt)*one_over_r(r)*ref%Lorentz_Coeff &
                - d2_m0(PSI2,dbrdrdt)*m0_values(PSI2,br)*one_over_r(r)*ref%Lorentz_Coeff &
                - d2_m0(PSI2,dbrdtdt)*m0_values(PSI2,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - m0_values(PSI2,dbrdr)*m0_values(PSI2,dbrdt)*one_over_r(r)*ref%Lorentz_Coeff &
                - m0_values(PSI2,dbrdt)*m0_values(PSI2,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + 2*m0_values(PSI2,bphi)*m0_values(PSI2,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + 2*m0_values(PSI2,br)*m0_values(PSI2,dbtdr)

## 12. `curl_vp_grad_vm_*` and `curl_vm_grad_vp_*`: asymmetric cross-advection

$(A\cdot\nabla)B$ is **not** symmetric in $A$ and $B$ — only $B$ is ever
differentiated (see `ADotGradB_3D3D` in `Diagnostics_ADotGradB.F90`). So
$v'\cdot\nabla\overline v$ and $\overline v\cdot\nabla v'$ are two genuinely
distinct quantities, each needing its own derivation — neither is a
substitution on the other, nor on the self-advection cases above.

In [50]:

def ADotGradB(Ar, At, Ap, Br, Bt, Bp):
    # Mirrors ADotGradB_3D3D exactly: only B is ever differentiated.
    dBrdr, dBrdt, dBrdp = sp.diff(Br,r), sp.diff(Br,theta), sp.diff(Br,phi)
    dBtdr, dBtdt, dBtdp = sp.diff(Bt,r), sp.diff(Bt,theta), sp.diff(Bt,phi)
    dBpdr, dBpdt, dBpdp = sp.diff(Bp,r), sp.diff(Bp,theta), sp.diff(Bp,phi)
    Cr = Ar*dBrdr + oor*(At*(dBrdt - Bt) + Ap*(csc*dBrdp - Bp))
    Ct = Ar*dBtdr + oor*(At*(dBtdt + Br) + Ap*(csc*dBtdp - cot*Bp))
    Cp = Ar*dBpdr + oor*(At*dBpdt + Ap*(csc*dBpdp + Br + cot*Bt))
    return Cr, Ct, Cp

# v' . grad <v>:  A = v' (fluctuating), B = <v> (mean)
Ar_pm, At_pm, Ap_pm = ADotGradB(vr, vt, vp, vmr, vmt, vmp)
Fr_pm, Ft_pm, Fp_pm = rho*Ar_pm, rho*At_pm, rho*Ap_pm
cr_pm, ct_pm, cp_pm = curl(Fr_pm, Ft_pm, Fp_pm)

# <v> . grad v':  A = <v> (mean), B = v' (fluctuating)
Ar_mp, At_mp, Ap_mp = ADotGradB(vmr, vmt, vmp, vr, vt, vp)
Fr_mp, Ft_mp, Fp_mp = rho*Ar_mp, rho*At_mp, rho*Ap_mp
cr_mp, ct_mp, cp_mp = curl(Fr_mp, Ft_mp, Fp_mp)

cr_pm, ct_pm, cp_pm


⎛⎛                                              ∂                             
⎜⎜                 2              r⋅vᵣ(r, θ, φ)⋅──(vₚ(r, θ))                  
⎜⎜                ∂                             ∂r               ∂            
⎜⎜r⋅vᵣ(r, θ, φ)⋅─────(vₚ(r, θ)) + ────────────────────────── + r⋅──(vₚ(r, θ))⋅
⎜⎜              ∂θ ∂r                       tan(θ)               ∂r           
⎜⎝                                                                            
⎜─────────────────────────────────────────────────────────────────────────────
⎜                                                                             
⎝                                                                             

                    ∂               ∂                              ∂          
                  r⋅──(vᵣ(r, θ, φ))⋅──(vₜ(r, θ))   vₚ(r, θ)⋅cos(θ)⋅──(vₚ(r, θ,
∂                   ∂φ              ∂r                             ∂φ         
──(vᵣ(r, θ, φ)) - ────────────────────────────── + 

### Fortranize both

Mixed fluctuating/mean expressions: `fbuffer`/`d2_fbuffer` for the $v'$
pieces, `m0_values`/`d2_m0` for the $\overline v$ pieces, in the same
formula.

In [51]:

def build_mixed_subs_list(pfields, mfields):
    field_buffer_name = {'vr': 'vr', 'vt': 'vtheta', 'vp': 'vphi'}
    varsym3 = {'r': r, 't': theta, 'p': phi}
    varsym2 = {'r': r, 't': theta}
    pairs3 = [('r','r'), ('r','t'), ('r','p'), ('t','t'), ('t','p'), ('p','p')]
    pairs2 = [('r','r'), ('r','t'), ('t','t')]
    subs_list = []
    for fshort, ffunc in pfields.items():
        for a, b in pairs3:
            subs_list.append((sp.diff(ffunc, varsym3[a], varsym3[b]),
                               sp.Symbol(f'd2_fbuffer(PSI,d{fshort}d{a}d{b})')))
        for a in ['r','t','p']:
            subs_list.append((sp.diff(ffunc, varsym3[a]), sp.Symbol(f'fbuffer(PSI,d{fshort}d{a})')))
        subs_list.append((ffunc, sp.Symbol(f'fbuffer(PSI,{field_buffer_name[fshort]})')))
    for fshort, ffunc in mfields.items():
        for a, b in pairs2:
            subs_list.append((sp.diff(ffunc, varsym2[a], varsym2[b]),
                               sp.Symbol(f'd2_m0(PSI2,d{fshort}d{a}d{b})')))
        for a in ['r','t']:
            subs_list.append((sp.diff(ffunc, varsym2[a]), sp.Symbol(f'm0_values(PSI2,d{fshort}d{a})')))
        subs_list.append((ffunc, sp.Symbol(f'm0_values(PSI2,{field_buffer_name[fshort]})')))
    return [(sp.diff(rho, r), DLNRHO*RHO)] + subs_list + [(rho, RHO)]

subs_list_pm = build_mixed_subs_list({'vr': vr, 'vt': vt, 'vp': vp}, {'vr': vmr, 'vt': vmt, 'vp': vmp})

def fortranize_mixed(expr, subs_list):
    e = expr.subs(subs_list)
    e = sp.expand(e)
    e = e.subs(metric_subs)
    e = sp.expand(e)
    return e

fr_out_pm = fortranize_mixed(cr_pm, subs_list_pm)
ft_out_pm = fortranize_mixed(ct_pm, subs_list_pm)
fp_out_pm = fortranize_mixed(cp_pm, subs_list_pm)

fr_out_mp = fortranize_mixed(cr_mp, subs_list_pm)
ft_out_mp = fortranize_mixed(ct_mp, subs_list_pm)
fp_out_mp = fortranize_mixed(cp_mp, subs_list_pm)

for name, e in [("curl_vp_grad_vm_r", fr_out_pm), ("curl_vp_grad_vm_theta", ft_out_pm), ("curl_vp_grad_vm_phi", fp_out_pm),
                ("curl_vm_grad_vp_r", fr_out_mp), ("curl_vm_grad_vp_theta", ft_out_mp), ("curl_vm_grad_vp_phi", fp_out_mp)]:
    residual_atoms = [a for a in e.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
    print(f"{name}: {'CLEAN (fully expressed in Fortran symbols)' if not residual_atoms else 'LEFTOVER: '+str(residual_atoms)}")


curl_vp_grad_vm_r: CLEAN (fully expressed in Fortran symbols)
curl_vp_grad_vm_theta: CLEAN (fully expressed in Fortran symbols)
curl_vp_grad_vm_phi: CLEAN (fully expressed in Fortran symbols)
curl_vm_grad_vp_r: CLEAN (fully expressed in Fortran symbols)
curl_vm_grad_vp_theta: CLEAN (fully expressed in Fortran symbols)
curl_vm_grad_vp_phi: CLEAN (fully expressed in Fortran symbols)


`curl_vp_grad_vm_r` (Fortran-symbol form)

In [52]:
print(fortran_lines(fr_out_pm))

                qty(PSI) = d2_m0(PSI2,dvpdrdt)*fbuffer(PSI,vr)*one_over_r(r)*ref%density(r) &
                + d2_m0(PSI2,dvpdtdt)*fbuffer(PSI,vtheta)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,dvpdt)*m0_values(PSI2,vr)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,dvrdt)*m0_values(PSI2,dvpdr)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvtdt)*m0_values(PSI2,dvpdt)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,vphi)*m0_values(PSI2,dvrdt)*one_over_r(r)**2*ref%density(r) &
                - fbuffer(PSI,vphi)*m0_values(PSI2,vtheta)*one_over_r(r)**2*ref%density(r) &
                + cottheta(t)*fbuffer(PSI,dvpdt)*m0_values(PSI2,vtheta)*one_over_r(r)**2*ref%density(r) &
                + cottheta(t)*fbuffer(PSI,vphi)*m0_values(PSI2,dvtdt)*one_over_r(r)**2*ref%density(r) &
                + cottheta(t)*fbuffer(PSI,vphi)*m0_values(PSI2,vr)*one_over_r(r)**2*ref%density(r) &
                + cottheta(t)*fbuffer(PSI,vr)

`curl_vp_grad_vm_theta` (Fortran-symbol form)

In [53]:
print(fortran_lines(ft_out_pm))

                qty(PSI) = -d2_m0(PSI2,dvpdrdr)*fbuffer(PSI,vr)*ref%density(r) &
                - fbuffer(PSI,dvrdr)*m0_values(PSI2,dvpdr)*ref%density(r) &
                - d2_m0(PSI2,dvpdrdt)*fbuffer(PSI,vtheta)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,dvpdr)*m0_values(PSI2,vr)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,dvtdr)*m0_values(PSI2,dvpdt)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,vphi)*m0_values(PSI2,dvrdr)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,vr)*m0_values(PSI2,dvpdr)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,vr)*m0_values(PSI2,dvpdr)*ref%density(r)*ref%dlnrho(r) &
                + csctheta(t)*fbuffer(PSI,dvrdp)*m0_values(PSI2,dvrdr)*one_over_r(r)*ref%density(r) &
                + csctheta(t)*fbuffer(PSI,dvtdp)*m0_values(PSI2,dvrdt)*one_over_r(r)**2*ref%density(r) &
                - cottheta(t)*fbuffer(PSI,dvpdr)*m0_values(PSI2,vtheta)*one_over_r(r)*ref%density(r) &
    

`curl_vp_grad_vm_phi` (Fortran-symbol form)

In [54]:
print(fortran_lines(fp_out_pm))

                qty(PSI) = d2_m0(PSI2,dvtdrdr)*fbuffer(PSI,vr)*ref%density(r) &
                + fbuffer(PSI,dvrdr)*m0_values(PSI2,dvtdr)*ref%density(r) &
                + d2_m0(PSI2,dvtdrdt)*fbuffer(PSI,vtheta)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvpdt)*m0_values(PSI2,vphi)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,dvtdr)*m0_values(PSI2,dvtdt)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvtdr)*m0_values(PSI2,vr)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvtdt)*m0_values(PSI2,vtheta)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,vphi)*m0_values(PSI2,dvpdt)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,vr)*m0_values(PSI2,dvtdr)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,vr)*m0_values(PSI2,dvtdr)*ref%density(r)*ref%dlnrho(r) &
                + fbuffer(PSI,vtheta)*m0_values(PSI2,dvrdr)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,vthet

`curl_vm_grad_vp_r` (Fortran-symbol form)

In [55]:
print(fortran_lines(fr_out_mp))

                qty(PSI) = d2_fbuffer(PSI,dvpdrdt)*m0_values(PSI2,vr)*one_over_r(r)*ref%density(r) &
                + d2_fbuffer(PSI,dvpdtdt)*m0_values(PSI2,vtheta)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,dvpdr)*m0_values(PSI2,dvrdt)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvpdt)*m0_values(PSI2,dvtdt)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,dvrdt)*m0_values(PSI2,vphi)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,vr)*m0_values(PSI2,dvpdt)*one_over_r(r)**2*ref%density(r) &
                - fbuffer(PSI,vtheta)*m0_values(PSI2,vphi)*one_over_r(r)**2*ref%density(r) &
                + cottheta(t)*fbuffer(PSI,dvpdr)*m0_values(PSI2,vr)*one_over_r(r)*ref%density(r) &
                + cottheta(t)*fbuffer(PSI,dvpdt)*m0_values(PSI2,vtheta)*one_over_r(r)**2*ref%density(r) &
                + cottheta(t)*fbuffer(PSI,dvtdt)*m0_values(PSI2,vphi)*one_over_r(r)**2*ref%density(r) &
                + cottheta(t)*fbu

`curl_vm_grad_vp_theta` (Fortran-symbol form)

In [56]:
print(fortran_lines(ft_out_mp))

                qty(PSI) = -d2_fbuffer(PSI,dvpdrdr)*m0_values(PSI2,vr)*ref%density(r) &
                - fbuffer(PSI,dvpdr)*m0_values(PSI2,dvrdr)*ref%density(r) &
                - d2_fbuffer(PSI,dvpdrdt)*m0_values(PSI2,vtheta)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,dvpdr)*m0_values(PSI2,vr)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,dvpdr)*m0_values(PSI2,vr)*ref%density(r)*ref%dlnrho(r) &
                - fbuffer(PSI,dvpdt)*m0_values(PSI2,dvtdr)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,dvrdr)*m0_values(PSI2,vphi)*one_over_r(r)*ref%density(r) &
                - fbuffer(PSI,vr)*m0_values(PSI2,dvpdr)*one_over_r(r)*ref%density(r) &
                + csctheta(t)*d2_fbuffer(PSI,dvrdrdp)*m0_values(PSI2,vr)*one_over_r(r)*ref%density(r) &
                + csctheta(t)*d2_fbuffer(PSI,dvrdtdp)*m0_values(PSI2,vtheta)*one_over_r(r)**2*ref%density(r) &
                + csctheta(t)**2*d2_fbuffer(PSI,dvrdpdp)*m0_values(PSI2,vphi)*one_ove

`curl_vm_grad_vp_phi` (Fortran-symbol form)

In [57]:
print(fortran_lines(fp_out_mp))

                qty(PSI) = d2_fbuffer(PSI,dvtdrdr)*m0_values(PSI2,vr)*ref%density(r) &
                + fbuffer(PSI,dvtdr)*m0_values(PSI2,dvrdr)*ref%density(r) &
                + d2_fbuffer(PSI,dvtdrdt)*m0_values(PSI2,vtheta)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvpdt)*m0_values(PSI2,vphi)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,dvrdr)*m0_values(PSI2,vtheta)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvtdr)*m0_values(PSI2,vr)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvtdr)*m0_values(PSI2,vr)*ref%density(r)*ref%dlnrho(r) &
                + fbuffer(PSI,dvtdt)*m0_values(PSI2,dvtdr)*one_over_r(r)*ref%density(r) &
                + fbuffer(PSI,dvtdt)*m0_values(PSI2,vtheta)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,vphi)*m0_values(PSI2,dvpdt)*one_over_r(r)**2*ref%density(r) &
                + fbuffer(PSI,vr)*m0_values(PSI2,dvtdr)*one_over_r(r)*ref%density(r) &
                + fbu

### Bonus consistency check: the four pieces sum to the whole

Reynolds decomposition is exact ($v=\overline v+v'$), and both `ADotGradB`
and $\nabla\times$ are linear/bilinear, so as an *operator identity*:
$$\nabla\times\left[\rho\,(v\cdot\nabla)v\right] =
\nabla\times\left[\rho\,(v'\cdot\nabla)v'\right] +
\nabla\times\left[\rho\,(v'\cdot\nabla)\overline v\right] +
\nabla\times\left[\rho\,(\overline v\cdot\nabla)v'\right] +
\nabla\times\left[\rho\,(\overline v\cdot\nabla)\overline v\right]$$
Check this directly and symbolically: build $v=v'+\overline v$ from
scratch and confirm `curl(rho*(v.grad)v)` computed that way exactly equals
the sum of `cr_a` (full, Section 5), `cr_pp`+`cr_mm` (self terms, Sections 7
and 10) and `cr_pm`+`cr_mp` (cross terms, this section) — a single check
that ties together every advection-force formula in this notebook.

In [58]:

vr_sum, vt_sum, vp_sum = vr + vmr, vt + vmt, vp + vmp
vgv_sum_r, vgv_sum_t, vgv_sum_p = ADotGradB(vr_sum, vt_sum, vp_sum, vr_sum, vt_sum, vp_sum)
Fr_sum, Ft_sum, Fp_sum = rho*vgv_sum_r, rho*vgv_sum_t, rho*vgv_sum_p
cr_sum, ct_sum, cp_sum = curl(Fr_sum, Ft_sum, Fp_sum)

# self terms from Sections 5 (full/reference identity), 7 (v'.grad v'), 10 (<v>.grad <v>)
Ar_pp, At_pp, Ap_pp = ADotGradB(vr, vt, vp, vr, vt, vp)
cr_pp, ct_pp, cp_pp = curl(rho*Ar_pp, rho*At_pp, rho*Ap_pp)

for name, whole, pieces in [
    ("r", cr_sum, cr_pp + cr_pm + cr_mp + cr_m),
    ("theta", ct_sum, ct_pp + ct_pm + ct_mp + ct_m),
    ("phi", cp_sum, cp_pp + cp_pm + cp_mp + cp_m),
]:
    print(f"{name}: curl(v.grad v) with v=v'+<v> equals sum of the four decomposed pieces = "
          f"{sp.simplify(whole - pieces) == 0}")


r: curl(v.grad v) with v=v'+<v> equals sum of the four decomposed pieces = True
theta: curl(v.grad v) with v=v'+<v> equals sum of the four decomposed pieces = True
phi: curl(v.grad v) with v=v'+<v> equals sum of the four decomposed pieces = True


## 13. `curl_jp_cross_bm_*` and `curl_jm_cross_bp_*`: asymmetric cross-Lorentz-force

Same asymmetry as Section 12, applied to the magnetic force: $L_c(\nabla
\times B')\times\overline B$ and $L_c(\nabla\times\overline B)\times B'$
are distinct — the curl is taken of one field, then crossed with the
*other*.

In [59]:

Jr_p, Jt_p, Jp_p = curl(Br, Bt, Bp)     # curl of fluctuating B (= J')
Fr_pbm = Lc*(Jt_p*Bmp - Jp_p*Bmt)
Ft_pbm = Lc*(Jp_p*Bmr - Jr_p*Bmp)
Fp_pbm = Lc*(Jr_p*Bmt - Jt_p*Bmr)
cr_pbm, ct_pbm, cp_pbm = curl(Fr_pbm, Ft_pbm, Fp_pbm)

Jr_m, Jt_m, Jp_m = curl(Bmr, Bmt, Bmp)  # curl of mean B (= <J>)
Fr_mbp = Lc*(Jt_m*Bp - Jp_m*Bt)
Ft_mbp = Lc*(Jp_m*Br - Jr_m*Bp)
Fp_mbp = Lc*(Jr_m*Bt - Jt_m*Br)
cr_mbp, ct_mbp, cp_mbp = curl(Fr_mbp, Ft_mbp, Fp_mbp)

cr_pbm, ct_pbm, cp_pbm


⎛    ⎛                                                                        
⎜    ⎜                                                                        
⎜    ⎜                                           ∂                            
⎜    ⎜              2                 r⋅Bᵣ(r, θ)⋅──(Bₚ(r, θ, φ))   r⋅Bᵣ(r, θ)⋅
⎜    ⎜             ∂                             ∂r                           
⎜L_c⋅⎜r⋅Bᵣ(r, θ)⋅─────(Bₚ(r, θ, φ)) + ────────────────────────── - ───────────
⎜    ⎜           ∂θ ∂r                          tan(θ)                        
⎜    ⎝                                                                        
⎜─────────────────────────────────────────────────────────────────────────────
⎜                                                                             
⎝                                                                             

                                                                              
   2                                               

### Fortranize both

In [60]:

def build_mixed_subs_list_b(pfields, mfields):
    field_buffer_name = {'br': 'br', 'bt': 'btheta', 'bp': 'bphi'}
    varsym3 = {'r': r, 't': theta, 'p': phi}
    varsym2 = {'r': r, 't': theta}
    pairs3 = [('r','r'), ('r','t'), ('r','p'), ('t','t'), ('t','p'), ('p','p')]
    pairs2 = [('r','r'), ('r','t'), ('t','t')]
    subs_list = []
    for fshort, ffunc in pfields.items():
        for a, b in pairs3:
            subs_list.append((sp.diff(ffunc, varsym3[a], varsym3[b]),
                               sp.Symbol(f'd2_fbuffer(PSI,d{fshort}d{a}d{b})')))
        for a in ['r','t','p']:
            subs_list.append((sp.diff(ffunc, varsym3[a]), sp.Symbol(f'fbuffer(PSI,d{fshort}d{a})')))
        subs_list.append((ffunc, sp.Symbol(f'fbuffer(PSI,{field_buffer_name[fshort]})')))
    for fshort, ffunc in mfields.items():
        for a, b in pairs2:
            subs_list.append((sp.diff(ffunc, varsym2[a], varsym2[b]),
                               sp.Symbol(f'd2_m0(PSI2,d{fshort}d{a}d{b})')))
        for a in ['r','t']:
            subs_list.append((sp.diff(ffunc, varsym2[a]), sp.Symbol(f'm0_values(PSI2,d{fshort}d{a})')))
        subs_list.append((ffunc, sp.Symbol(f'm0_values(PSI2,{field_buffer_name[fshort]})')))
    return subs_list

subs_list_jpm = build_mixed_subs_list_b({'br': Br, 'bt': Bt, 'bp': Bp}, {'br': Bmr, 'bt': Bmt, 'bp': Bmp})

def fortranize_mixed_b(expr, subs_list):
    e = expr.subs(subs_list)
    e = sp.expand(e)
    e = e.subs(metric_subs_jm)
    e = sp.expand(e)
    return e

fr_out_pbm = fortranize_mixed_b(cr_pbm, subs_list_jpm)
ft_out_pbm = fortranize_mixed_b(ct_pbm, subs_list_jpm)
fp_out_pbm = fortranize_mixed_b(cp_pbm, subs_list_jpm)

fr_out_mbp = fortranize_mixed_b(cr_mbp, subs_list_jpm)
ft_out_mbp = fortranize_mixed_b(ct_mbp, subs_list_jpm)
fp_out_mbp = fortranize_mixed_b(cp_mbp, subs_list_jpm)

for name, e in [("curl_jp_cross_bm_r", fr_out_pbm), ("curl_jp_cross_bm_theta", ft_out_pbm), ("curl_jp_cross_bm_phi", fp_out_pbm),
                ("curl_jm_cross_bp_r", fr_out_mbp), ("curl_jm_cross_bp_theta", ft_out_mbp), ("curl_jm_cross_bp_phi", fp_out_mbp)]:
    residual_atoms = [a for a in e.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
    print(f"{name}: {'CLEAN (fully expressed in Fortran symbols)' if not residual_atoms else 'LEFTOVER: '+str(residual_atoms)}")


curl_jp_cross_bm_r: CLEAN (fully expressed in Fortran symbols)
curl_jp_cross_bm_theta: CLEAN (fully expressed in Fortran symbols)
curl_jp_cross_bm_phi: CLEAN (fully expressed in Fortran symbols)
curl_jm_cross_bp_r: CLEAN (fully expressed in Fortran symbols)
curl_jm_cross_bp_theta: CLEAN (fully expressed in Fortran symbols)
curl_jm_cross_bp_phi: CLEAN (fully expressed in Fortran symbols)


`curl_jp_cross_bm_r` (Fortran-symbol form)

In [61]:
print(fortran_lines(fr_out_pbm))

                qty(PSI) = d2_fbuffer(PSI,dbpdrdt)*m0_values(PSI2,br)*one_over_r(r)*ref%Lorentz_Coeff &
                + d2_fbuffer(PSI,dbpdtdt)*m0_values(PSI2,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,bphi)*m0_values(PSI2,dbrdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbpdr)*m0_values(PSI2,dbrdt)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbpdt)*m0_values(PSI2,br)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbpdt)*m0_values(PSI2,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - fbuffer(PSI,bphi)*m0_values(PSI2,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + cottheta(t)*fbuffer(PSI,bphi)*m0_values(PSI2,br)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + cottheta(t)*fbuffer(PSI,bphi)*m0_values(PSI2,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + cottheta(t)*fbuffer(PSI,dbpdr)*m0_values(PSI2,br)*one_over_r(r)*ref%Lorentz_Coeff &
        

`curl_jp_cross_bm_theta` (Fortran-symbol form)

In [62]:
print(fortran_lines(ft_out_pbm))

                qty(PSI) = -d2_fbuffer(PSI,dbpdrdr)*m0_values(PSI2,br)*ref%Lorentz_Coeff &
                - fbuffer(PSI,dbpdr)*m0_values(PSI2,dbrdr)*ref%Lorentz_Coeff &
                - d2_fbuffer(PSI,dbpdrdt)*m0_values(PSI2,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                - fbuffer(PSI,bphi)*m0_values(PSI2,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - fbuffer(PSI,dbpdt)*m0_values(PSI2,dbtdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - 2*fbuffer(PSI,dbpdr)*m0_values(PSI2,br)*one_over_r(r)*ref%Lorentz_Coeff &
                + csctheta(t)*d2_fbuffer(PSI,dbrdrdp)*m0_values(PSI2,br)*one_over_r(r)*ref%Lorentz_Coeff &
                + csctheta(t)*d2_fbuffer(PSI,dbrdtdp)*m0_values(PSI2,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + csctheta(t)*fbuffer(PSI,dbrdp)*m0_values(PSI2,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + csctheta(t)*fbuffer(PSI,dbtdp)*m0_values(PSI2,dbtdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + cscth

`curl_jp_cross_bm_phi` (Fortran-symbol form)

In [63]:
print(fortran_lines(fp_out_pbm))

                qty(PSI) = d2_fbuffer(PSI,dbtdrdr)*m0_values(PSI2,br)*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbtdr)*m0_values(PSI2,dbrdr)*ref%Lorentz_Coeff &
                + d2_fbuffer(PSI,dbtdrdt)*m0_values(PSI2,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,bphi)*m0_values(PSI2,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,btheta)*m0_values(PSI2,dbrdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,btheta)*m0_values(PSI2,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbpdr)*m0_values(PSI2,dbpdt)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbpdt)*m0_values(PSI2,bphi)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbtdr)*m0_values(PSI2,dbtdt)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbtdt)*m0_values(PSI2,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - d2_fbuffer(PSI,dbrdrdt)*m0_values(PSI2,br)*one_ove

`curl_jm_cross_bp_r` (Fortran-symbol form)

In [64]:
print(fortran_lines(fr_out_mbp))

                qty(PSI) = d2_m0(PSI2,dbpdrdt)*fbuffer(PSI,br)*one_over_r(r)*ref%Lorentz_Coeff &
                + d2_m0(PSI2,dbpdtdt)*fbuffer(PSI,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,br)*m0_values(PSI2,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbrdt)*m0_values(PSI2,bphi)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbrdt)*m0_values(PSI2,dbpdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbtdt)*m0_values(PSI2,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - fbuffer(PSI,btheta)*m0_values(PSI2,bphi)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + cottheta(t)*fbuffer(PSI,br)*m0_values(PSI2,bphi)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + cottheta(t)*fbuffer(PSI,br)*m0_values(PSI2,dbpdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + cottheta(t)*fbuffer(PSI,dbtdt)*m0_values(PSI2,bphi)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + csct

`curl_jm_cross_bp_theta` (Fortran-symbol form)

In [65]:
print(fortran_lines(ft_out_mbp))

                qty(PSI) = -d2_m0(PSI2,dbpdrdr)*fbuffer(PSI,br)*ref%Lorentz_Coeff &
                - fbuffer(PSI,dbrdr)*m0_values(PSI2,dbpdr)*ref%Lorentz_Coeff &
                - d2_m0(PSI2,dbpdrdt)*fbuffer(PSI,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                - fbuffer(PSI,dbrdr)*m0_values(PSI2,bphi)*one_over_r(r)*ref%Lorentz_Coeff &
                - fbuffer(PSI,dbtdr)*m0_values(PSI2,dbpdt)*one_over_r(r)*ref%Lorentz_Coeff &
                - 2*fbuffer(PSI,br)*m0_values(PSI2,dbpdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + csctheta(t)*fbuffer(PSI,dbtdp)*m0_values(PSI2,dbrdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - cottheta(t)*fbuffer(PSI,btheta)*m0_values(PSI2,dbpdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - cottheta(t)*fbuffer(PSI,dbtdr)*m0_values(PSI2,bphi)*one_over_r(r)*ref%Lorentz_Coeff &
                - csctheta(t)*fbuffer(PSI,dbpdp)*m0_values(PSI2,bphi)*one_over_r(r)**2*ref%Lorentz_Coeff &
                - csctheta(t)*fbuffer(PSI,d

`curl_jm_cross_bp_phi` (Fortran-symbol form)

In [66]:
print(fortran_lines(fp_out_mbp))

                qty(PSI) = d2_m0(PSI2,dbtdrdr)*fbuffer(PSI,br)*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbrdr)*m0_values(PSI2,dbtdr)*ref%Lorentz_Coeff &
                + d2_m0(PSI2,dbtdrdt)*fbuffer(PSI,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,bphi)*m0_values(PSI2,dbpdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,btheta)*m0_values(PSI2,dbtdt)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbpdt)*m0_values(PSI2,bphi)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbpdt)*m0_values(PSI2,dbpdr)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbrdr)*m0_values(PSI2,btheta)*one_over_r(r)*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbtdt)*m0_values(PSI2,btheta)*one_over_r(r)**2*ref%Lorentz_Coeff &
                + fbuffer(PSI,dbtdt)*m0_values(PSI2,dbtdr)*one_over_r(r)*ref%Lorentz_Coeff &
                - d2_m0(PSI2,dbrdrdt)*fbuffer(PSI,br)*one_over_r(r)*ref%Lorentz_Co

### Bonus consistency check: the four pieces sum to the whole

Same identity as Section 12's, applied to the Lorentz force: $B=B'+\overline
B$ exactly, and $(\nabla\times\cdot)\times\cdot$ is bilinear in its two
factors, so `curl(Lc*(curl B) x B)` built from $B=B'+\overline B$ from
scratch should equal the sum of Section 6 (full, `cr_j`), Sections 9 and 11
(self terms, `cr_j_pp`/`cr_jm`) and this section's two cross terms
(`cr_pbm`/`cr_mbp`).

In [67]:

Br_sum, Bt_sum, Bp_sum = Br + Bmr, Bt + Bmt, Bp + Bmp
J_sum_r, J_sum_t, J_sum_p = curl(Br_sum, Bt_sum, Bp_sum)
Fr_jsum = Lc*(J_sum_t*Bp_sum - J_sum_p*Bt_sum)
Ft_jsum = Lc*(J_sum_p*Br_sum - J_sum_r*Bp_sum)
Fp_jsum = Lc*(J_sum_r*Bt_sum - J_sum_t*Br_sum)
cr_jsum, ct_jsum, cp_jsum = curl(Fr_jsum, Ft_jsum, Fp_jsum)

Jr_pp, Jt_pp, Jp_pp = curl(Br, Bt, Bp)
Fr_j_pp = Lc*(Jt_pp*Bp - Jp_pp*Bt)
Ft_j_pp = Lc*(Jp_pp*Br - Jr_pp*Bp)
Fp_j_pp = Lc*(Jr_pp*Bt - Jt_pp*Br)
cr_j_pp, ct_j_pp, cp_j_pp = curl(Fr_j_pp, Ft_j_pp, Fp_j_pp)

for name, whole, pieces in [
    ("r", cr_jsum, cr_j_pp + cr_pbm + cr_mbp + cr_jm),
    ("theta", ct_jsum, ct_j_pp + ct_pbm + ct_mbp + ct_jm),
    ("phi", cp_jsum, cp_j_pp + cp_pbm + cp_mbp + cp_jm),
]:
    print(f"{name}: curl(Lc*(curl B) x B) with B=B'+<B> equals sum of the four decomposed pieces = "
          f"{sp.simplify(whole - pieces) == 0}")


r: curl(Lc*(curl B) x B) with B=B'+<B> equals sum of the four decomposed pieces = True
theta: curl(Lc*(curl B) x B) with B=B'+<B> equals sum of the four decomposed pieces = True
phi: curl(Lc*(curl B) x B) with B=B'+<B> equals sum of the four decomposed pieces = True


## 14. `curl_buoyancy_pforce_*` and `curl_buoyancy_mforce_*`

Same structural formula as Section 2 (`curl_buoyancy_force_*`), with
$\Theta\to\Theta'$ (fluctuating) or $\Theta\to\overline\Theta$ (mean).
$\overline\Theta$ is axisymmetric, so `curl_buoyancy_mforce_theta`
$\propto\partial_\phi\overline\Theta$ is identically 0 — no code is defined
for it in `curl_momentum_equation_codes.F` (the offset is left as a gap,
matching the existing convention for `curl_buoyancy_force_r`/
`curl_buoyancy_pforce_r`, which are also always 0).

In [68]:

Tp = sp.Function('Tp')(r, theta, phi)
Tm = sp.Function('Tm')(r, theta)

cr_bp, ct_bp, cp_bp = curl(Bc*Tp, sp.Integer(0), sp.Integer(0))
cr_bm, ct_bm, cp_bm = curl(Bc*Tm, sp.Integer(0), sp.Integer(0))
print("curl_r (pforce, true):", cr_bp, " -- always 0, as expected")
print("curl_r (mforce, true):", cr_bm, " -- always 0, as expected")
print("curl_theta (mforce, true):", ct_bm, " -- identically 0: <Theta> is axisymmetric")

BCOEFF = sp.Symbol('ref%Buoyancy_Coeff(r)')
subs_bp = [(sp.diff(Tp, theta), sp.Symbol('fbuffer(PSI,dtdt)')),
           (sp.diff(Tp, phi), sp.Symbol('fbuffer(PSI,dtdp)')),
           (Tp, sp.Symbol('fbuffer(PSI,tvar)')), (Bc, BCOEFF)]
subs_bm = [(sp.diff(Tm, theta), sp.Symbol('m0_values(PSI2,dtdt)')),
           (Tm, sp.Symbol('m0_values(PSI2,tvar)')), (Bc, BCOEFF)]

ft_out_bp = sp.expand(ct_bp.subs(subs_bp)).subs([(1/r, OOR), (1/sp.sin(theta), CSC)])
fp_out_bp = sp.expand(cp_bp.subs(subs_bp)).subs([(1/r, OOR), (1/sp.sin(theta), CSC)])
fp_out_bm = sp.expand(cp_bm.subs(subs_bm)).subs([(1/r, OOR), (1/sp.sin(theta), CSC)])

for name, e in [("curl_buoyancy_pforce_theta", ft_out_bp), ("curl_buoyancy_pforce_phi", fp_out_bp),
                ("curl_buoyancy_mforce_phi", fp_out_bm)]:
    residual_atoms = [a for a in e.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
    print(f"{name}: {'CLEAN (fully expressed in Fortran symbols)' if not residual_atoms else 'LEFTOVER: '+str(residual_atoms)}")


curl_r (pforce, true): 0  -- always 0, as expected
curl_r (mforce, true): 0  -- always 0, as expected
curl_theta (mforce, true): 0  -- identically 0: <Theta> is axisymmetric
curl_buoyancy_pforce_theta: CLEAN (fully expressed in Fortran symbols)
curl_buoyancy_pforce_phi: CLEAN (fully expressed in Fortran symbols)
curl_buoyancy_mforce_phi: CLEAN (fully expressed in Fortran symbols)


`curl_buoyancy_pforce_theta` (Fortran-symbol form)

In [69]:
print(fortran_lines(ft_out_bp))

                qty(PSI) = csctheta(t)*fbuffer(PSI,dtdp)*one_over_r(r)*ref%Buoyancy_Coeff(r)


`curl_buoyancy_pforce_phi` (Fortran-symbol form)

In [70]:
print(fortran_lines(fp_out_bp))

                qty(PSI) = -fbuffer(PSI,dtdt)*one_over_r(r)*ref%Buoyancy_Coeff(r)


`curl_buoyancy_mforce_phi` (Fortran-symbol form)

In [71]:
print(fortran_lines(fp_out_bm))

                qty(PSI) = -m0_values(PSI2,dtdt)*one_over_r(r)*ref%Buoyancy_Coeff(r)


## 15. `curl_coriolis_pforce_*` and `curl_coriolis_mforce_*`

Same structural formula as Section 1 (`curl_coriolis_force_*`), with
$v\to v'$ or $v\to\overline v$. Coriolis is *linear* in $v$ (not a
gradient), so — unlike buoyancy and pressure — there is no identically-zero
component here for either decomposition; all six pieces are genuinely
needed.

In [72]:

cr_cp, ct_cp, cp_cp = curl(C*rho*sp.sin(theta)*vp, C*rho*sp.cos(theta)*vp,
                            -C*rho*(sp.cos(theta)*vt + sp.sin(theta)*vr))
cr_cm, ct_cm, cp_cm = curl(C*rho*sp.sin(theta)*vmp, C*rho*sp.cos(theta)*vmp,
                            -C*rho*(sp.cos(theta)*vmt + sp.sin(theta)*vmr))

def build_full_v_subs(vr_, vt_, vp_, buf, is2d):
    varsym = {'r': r, 't': theta} if is2d else {'r': r, 't': theta, 'p': phi}
    idxname = 'PSI2' if is2d else 'PSI'
    field_buffer_name = {'vr': 'vr', 'vt': 'vtheta', 'vp': 'vphi'}
    fields = {'vr': vr_, 'vt': vt_, 'vp': vp_}
    subs = []
    for short, func in fields.items():
        for a in varsym:
            subs.append((sp.diff(func, varsym[a]), sp.Symbol(f'{buf}({idxname},d{short}d{a})')))
        subs.append((func, sp.Symbol(f'{buf}({idxname},{field_buffer_name[short]})')))
    subs = [(sp.diff(rho, r), DLNRHO*RHO)] + subs + [(rho, RHO), (C, sp.Symbol('ref%Coriolis_Coeff'))]
    return subs

def fortranize_coriolis(expr, subs):
    e = sp.expand(expr.subs(subs))
    e = e.subs([(sp.cos(theta), COS_), (sp.sin(theta), SIN_), (1/r, OOR),
                (sp.cos(theta)/sp.sin(theta), COT), (1/sp.sin(theta), CSC),
                (1/sp.tan(theta), COT), (sp.tan(theta), 1/COT)])
    return sp.expand(e)

subs_cp = build_full_v_subs(vr, vt, vp, 'fbuffer', is2d=False)
subs_cm = build_full_v_subs(vmr, vmt, vmp, 'm0_values', is2d=True)

cr_cp_f, ct_cp_f, cp_cp_f = (fortranize_coriolis(e, subs_cp) for e in (cr_cp, ct_cp, cp_cp))
cr_cm_f, ct_cm_f, cp_cm_f = (fortranize_coriolis(e, subs_cm) for e in (cr_cm, ct_cm, cp_cm))

for name, e in [("curl_coriolis_pforce_r", cr_cp_f), ("curl_coriolis_pforce_theta", ct_cp_f), ("curl_coriolis_pforce_phi", cp_cp_f),
                ("curl_coriolis_mforce_r", cr_cm_f), ("curl_coriolis_mforce_theta", ct_cm_f), ("curl_coriolis_mforce_phi", cp_cm_f)]:
    residual_atoms = [a for a in e.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
    print(f"{name}: {'CLEAN (fully expressed in Fortran symbols)' if not residual_atoms else 'LEFTOVER: '+str(residual_atoms)}")


curl_coriolis_pforce_r: CLEAN (fully expressed in Fortran symbols)
curl_coriolis_pforce_theta: CLEAN (fully expressed in Fortran symbols)
curl_coriolis_pforce_phi: CLEAN (fully expressed in Fortran symbols)
curl_coriolis_mforce_r: CLEAN (fully expressed in Fortran symbols)
curl_coriolis_mforce_theta: CLEAN (fully expressed in Fortran symbols)
curl_coriolis_mforce_phi: CLEAN (fully expressed in Fortran symbols)


`curl_coriolis_pforce_r` (Fortran-symbol form)

In [73]:
print(fortran_lines(cr_cp_f))

                qty(PSI) = -costheta(t)*fbuffer(PSI,dvtdt)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r) &
                - cottheta(t)*fbuffer(PSI,dvpdp)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r) &
                - fbuffer(PSI,dvrdt)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r)*sintheta(t) &
                - fbuffer(PSI,vtheta)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r)/sintheta(t) &
                - 2*costheta(t)*fbuffer(PSI,vr)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r) &
                + 2*fbuffer(PSI,vtheta)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r)*sintheta(t)


`curl_coriolis_pforce_theta` (Fortran-symbol form)

In [74]:
print(fortran_lines(ct_cp_f))

                qty(PSI) = costheta(t)*fbuffer(PSI,dvtdr)*ref%Coriolis_Coeff*ref%density(r) &
                + fbuffer(PSI,dvpdp)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r) &
                + fbuffer(PSI,dvrdr)*ref%Coriolis_Coeff*ref%density(r)*sintheta(t) &
                + costheta(t)*fbuffer(PSI,vtheta)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r) &
                + costheta(t)*fbuffer(PSI,vtheta)*ref%Coriolis_Coeff*ref%density(r)*ref%dlnrho(r) &
                + fbuffer(PSI,vr)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r)*sintheta(t) &
                + fbuffer(PSI,vr)*ref%Coriolis_Coeff*ref%density(r)*ref%dlnrho(r)*sintheta(t)


`curl_coriolis_pforce_phi` (Fortran-symbol form)

In [75]:
print(fortran_lines(cp_cp_f))

                qty(PSI) = costheta(t)*fbuffer(PSI,dvpdr)*ref%Coriolis_Coeff*ref%density(r) &
                + costheta(t)*fbuffer(PSI,vphi)*ref%Coriolis_Coeff*ref%density(r)*ref%dlnrho(r) &
                - fbuffer(PSI,dvpdt)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r)*sintheta(t)


`curl_coriolis_mforce_r` (Fortran-symbol form)

In [76]:
print(fortran_lines(cr_cm_f))

                qty(PSI) = -costheta(t)*m0_values(PSI2,dvtdt)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r) &
                - m0_values(PSI2,dvrdt)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r)*sintheta(t) &
                - m0_values(PSI2,vtheta)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r)/sintheta(t) &
                - 2*costheta(t)*m0_values(PSI2,vr)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r) &
                + 2*m0_values(PSI2,vtheta)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r)*sintheta(t)


`curl_coriolis_mforce_theta` (Fortran-symbol form)

In [77]:
print(fortran_lines(ct_cm_f))

                qty(PSI) = costheta(t)*m0_values(PSI2,dvtdr)*ref%Coriolis_Coeff*ref%density(r) &
                + m0_values(PSI2,dvrdr)*ref%Coriolis_Coeff*ref%density(r)*sintheta(t) &
                + costheta(t)*m0_values(PSI2,vtheta)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r) &
                + costheta(t)*m0_values(PSI2,vtheta)*ref%Coriolis_Coeff*ref%density(r)*ref%dlnrho(r) &
                + m0_values(PSI2,vr)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r)*sintheta(t) &
                + m0_values(PSI2,vr)*ref%Coriolis_Coeff*ref%density(r)*ref%dlnrho(r)*sintheta(t)


`curl_coriolis_mforce_phi` (Fortran-symbol form)

In [78]:
print(fortran_lines(cp_cm_f))

                qty(PSI) = costheta(t)*m0_values(PSI2,dvpdr)*ref%Coriolis_Coeff*ref%density(r) &
                + costheta(t)*m0_values(PSI2,vphi)*ref%Coriolis_Coeff*ref%density(r)*ref%dlnrho(r) &
                - m0_values(PSI2,dvpdt)*one_over_r(r)*ref%Coriolis_Coeff*ref%density(r)*sintheta(t)


## 16. `curl_pressure_pforce_*` and `curl_pressure_mforce_*`

Same structural formula as Section 3 (`curl_pressure_force_*`), with
$P\to P'$ or $P\to\overline P$. As with buoyancy, `curl_pressure_mforce_theta`
$\propto\partial_\phi\overline P$ is identically 0 (`<P>` is axisymmetric) —
no code is defined for it, matching `curl_pressure_force_r`/
`curl_pressure_pforce_r`'s gap convention.

In [79]:

Pp = sp.Function('Pp')(r, theta, phi)
Pm = sp.Function('Pm')(r, theta)

Fr_pp = -pfactor*sp.diff(Pp, r) + pfactor*dlnrho*Pp
Ft_pp = -pfactor*sp.diff(Pp, theta)/r
Fp_pp = -pfactor*sp.diff(Pp, phi)/(r*sp.sin(theta))
cr_prp, ct_prp, cp_prp = curl(Fr_pp, Ft_pp, Fp_pp)
print("curl_r (pforce, true):", cr_prp, " -- always 0, as expected")

Fr_pm = -pfactor*sp.diff(Pm, r) + pfactor*dlnrho*Pm
Ft_pm = -pfactor*sp.diff(Pm, theta)/r
Fp_pm = -pfactor*sp.diff(Pm, phi)/(r*sp.sin(theta))
cr_prm, ct_prm, cp_prm = curl(Fr_pm, Ft_pm, Fp_pm)
print("curl_r (mforce, true):", cr_prm, " -- always 0, as expected")
print("curl_theta (mforce, true):", ct_prm, " -- identically 0: <P> is axisymmetric")

subs_pp = [(sp.diff(Pp, theta), sp.Symbol('fbuffer(PSI,dpdt)')),
           (sp.diff(Pp, phi), sp.Symbol('fbuffer(PSI,dpdp)')), (dlnrho, DLNRHO)]
subs_pm = [(sp.diff(Pm, theta), sp.Symbol('m0_values(PSI2,dpdt)')), (dlnrho, DLNRHO)]

ft_out_prp = sp.expand(ct_prp.subs(subs_pp)).subs([(1/r, OOR), (1/sp.sin(theta), CSC)])
fp_out_prp = sp.expand(cp_prp.subs(subs_pp)).subs([(1/r, OOR), (1/sp.sin(theta), CSC)])
fp_out_prm = sp.expand(cp_prm.subs(subs_pm)).subs([(1/r, OOR), (1/sp.sin(theta), CSC)])

for name, e in [("curl_pressure_pforce_theta", ft_out_prp), ("curl_pressure_pforce_phi", fp_out_prp),
                ("curl_pressure_mforce_phi", fp_out_prm)]:
    residual_atoms = [a for a in e.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
    print(f"{name}: {'CLEAN (fully expressed in Fortran symbols)' if not residual_atoms else 'LEFTOVER: '+str(residual_atoms)}")


curl_r (pforce, true): 0  -- always 0, as expected
curl_r (mforce, true): 0  -- always 0, as expected
curl_theta (mforce, true): 0  -- identically 0: <P> is axisymmetric
curl_pressure_pforce_theta: CLEAN (fully expressed in Fortran symbols)
curl_pressure_pforce_phi: CLEAN (fully expressed in Fortran symbols)
curl_pressure_mforce_phi: CLEAN (fully expressed in Fortran symbols)


`curl_pressure_pforce_theta` (Fortran-symbol form)

In [80]:
print(fortran_lines(ft_out_prp))

                qty(PSI) = csctheta(t)*fbuffer(PSI,dpdp)*one_over_r(r)*p_f*ref%dlnrho(r)


`curl_pressure_pforce_phi` (Fortran-symbol form)

In [81]:
print(fortran_lines(fp_out_prp))

                qty(PSI) = -fbuffer(PSI,dpdt)*one_over_r(r)*p_f*ref%dlnrho(r)


`curl_pressure_mforce_phi` (Fortran-symbol form)

In [82]:
print(fortran_lines(fp_out_prm))

                qty(PSI) = -m0_values(PSI2,dpdt)*one_over_r(r)*p_f*ref%dlnrho(r)


## 17. `curl_buoyancy_force_abs`, `curl_coriolis_force_abs`, `curl_viscous_force_abs`

The three *full*-force magnitude diagnostics that predate this notebook
(`curl_v_grad_v_abs` and `curl_j_cross_b_abs` already got this treatment in
Sections 5 and 6). Each is coded directly from the already-verified
`r`/`theta`/`phi` component formulas above, so the check is: does
$\sqrt{\hat r^2+\hat\theta^2+\hat\phi^2}$, built from the exact bracket
expressions written in the `curl_*_abs` code, equal
$\sqrt{(\text{curl}_r)^2+(\text{curl}_\theta)^2+(\text{curl}_\phi)^2}$ using
the independently-verified `code_*` expressions from Sections 1, 2 and 4?
Comparing the squared magnitudes avoids any sign ambiguity from `sqrt`.

In [83]:

# --- curl_buoyancy_force_abs: codes one_over_r(r) * sqrt(bracket_theta**2 + bracket_phi**2)
# where bracket_theta = Bc*csctheta(t)*dtdp, bracket_phi = -Bc*dtdt (one_over_r
# factored out of the sqrt as a common factor of code_t_b/code_p_b).
bracket_t_b = Bc * csc * dtdp
bracket_p_b = -Bc * dtdt
abs_b_coded_sq = oor**2 * (bracket_t_b**2 + bracket_p_b**2)
abs_b_true_sq = code_t_b**2 + code_p_b**2
print("curl_buoyancy_force_abs: squared magnitude matches =",
      sp.simplify(sp.expand(abs_b_coded_sq - abs_b_true_sq)) == 0)

# --- curl_coriolis_force_abs: codes sqrt(bracket_r**2+bracket_t**2+bracket_p**2)
# directly from the same three bracket expressions as curl_coriolis_force_r/theta/phi.
dlnrho_true = sp.diff(rho, r)/rho  # dlnrho has been rebound (as an independent
                                    # Function) by later sections; recompute the
                                    # original Section-1 meaning locally so this
                                    # check isn't comparing against the wrong symbol.
bracket_r_cor = - C*rho*oor*(-sin_*vt + cot*cos_*vt + cos_*dvtdt + 2*cos_*vr + sin_*dvrdt + cot*dvpdp)
bracket_t_cor = C*rho*(oor*(dvpdp + cos_*vt + sin_*vr) + cos_*dvtdr + sin_*dvrdr + dlnrho_true*cos_*vt + dlnrho_true*sin_*vr)
bracket_p_cor = C*rho*(dlnrho_true*cos_*vp + cos_*dvpdr - oor*sin_*dvpdt)
abs_cor_coded_sq = bracket_r_cor**2 + bracket_t_cor**2 + bracket_p_cor**2
abs_cor_true_sq = code_r_cor**2 + code_t_cor**2 + code_p_cor**2
print("curl_coriolis_force_abs: squared magnitude matches =",
      sp.simplify(sp.expand(abs_cor_coded_sq - abs_cor_true_sq)) == 0)

# --- curl_viscous_force_abs: same pattern, sqrt(bracket_r**2+bracket_t**2+bracket_p**2)
# directly from the curl_viscous_force_r/theta/phi bracket expressions.
bracket_r_visc = oor*(sp.diff(vfp, theta) + cot*vfp - csc*sp.diff(vft, phi))
bracket_t_visc = oor*(csc*sp.diff(vfr, phi) - vfp) - sp.diff(vfp, r)
bracket_p_visc = sp.diff(vft, r) + oor*(vft - sp.diff(vfr, theta))
abs_visc_coded_sq = bracket_r_visc**2 + bracket_t_visc**2 + bracket_p_visc**2
abs_visc_true_sq = code_r_v**2 + code_t_v**2 + code_p_v**2
print("curl_viscous_force_abs: squared magnitude matches =",
      sp.simplify(sp.expand(abs_visc_coded_sq - abs_visc_true_sq)) == 0)


curl_buoyancy_force_abs: squared magnitude matches = True
curl_coriolis_force_abs: squared magnitude matches = True
curl_viscous_force_abs: squared magnitude matches = True


## 18. `curl_viscous_pforce_*` and `curl_viscous_mforce_*` (explicit check)

Section 9 argued by *citation* that `curl_viscous_force_*`'s generic
vector-curl formula (Section 4) applies verbatim to the fluctuating/mean
viscous force fields, since the derivation never assumed anything about
what the vector field physically represents — only that
`Grad_Viscous_Force` supplies its first derivatives. This section makes
that check explicit: substitute the pforce/mforce buffer-index names
(`vfp_r/t/p`, `dvfp_*_d*` and `vfm_r/t/p`, `dvfm_*_d*`) for the generic
`vfr`/`vft`/`vfp` derivatives used in Section 4, and print the result
directly for comparison against `Diagnostics_Curl_Momentum.F90`.

In [84]:

def relabel_viscous(expr, prefix):
    # prefix: 'p' or 'm'
    subs = {
        vfr: sp.Symbol(f'vforce_buffer(PSI,vf{prefix}_r)'),
        vft: sp.Symbol(f'vforce_buffer(PSI,vf{prefix}_t)'),
        vfp: sp.Symbol(f'vforce_buffer(PSI,vf{prefix}_p)'),
        sp.diff(vfr, theta): sp.Symbol(f'VFDBUFF(PSI,dvf{prefix}_r_dt)'),
        sp.diff(vfr, phi):   sp.Symbol(f'VFDBUFF(PSI,dvf{prefix}_r_dp)'),
        sp.diff(vft, r):     sp.Symbol(f'VFDBUFF(PSI,dvf{prefix}_t_dr)'),
        sp.diff(vft, phi):   sp.Symbol(f'VFDBUFF(PSI,dvf{prefix}_t_dp)'),
        sp.diff(vfp, r):     sp.Symbol(f'VFDBUFF(PSI,dvf{prefix}_p_dr)'),
        sp.diff(vfp, theta): sp.Symbol(f'VFDBUFF(PSI,dvf{prefix}_p_dt)'),
    }
    e = expr.subs(subs)
    e = sp.expand(e)
    e = e.subs(metric_subs)  # oor/csc/cot/cos_ -> one_over_r(r)/csctheta(t)/cottheta(t)/costheta(t)
    return sp.expand(e)

for label, prefix in [("pforce", "p"), ("mforce", "m")]:
    for comp, expr in [("r", code_r_v), ("theta", code_t_v), ("phi", code_p_v)]:
        relabeled = relabel_viscous(expr, prefix)
        residual_atoms = [a for a in relabeled.atoms(sp.Function, sp.Derivative) if not isinstance(a, sp.Symbol)]
        status = 'CLEAN' if not residual_atoms else f'LEFTOVER: {residual_atoms}'
        print(f"curl_viscous_{label}_{comp}: {status}")


curl_viscous_pforce_r: CLEAN
curl_viscous_pforce_theta: CLEAN
curl_viscous_pforce_phi: CLEAN
curl_viscous_mforce_r: CLEAN
curl_viscous_mforce_theta: CLEAN
curl_viscous_mforce_phi: CLEAN


`curl_viscous_pforce_r/theta/phi` and `curl_viscous_mforce_r/theta/phi` (Fortran-symbol form)

In [85]:

for label, prefix in [("pforce", "p"), ("mforce", "m")]:
    for comp, expr in [("r", code_r_v), ("theta", code_t_v), ("phi", code_p_v)]:
        print(f"--- curl_viscous_{label}_{comp} ---")
        print(fortran_lines(relabel_viscous(expr, prefix), lhs='qty(PSI) = '))
        print()


--- curl_viscous_pforce_r ---
                qty(PSI) = VFDBUFF(PSI,dvfp_p_dt)*one_over_r(r) - VFDBUFF(PSI,dvfp_t_dp)*csctheta(t)*one_over_r(r) &
                + costheta(t)*csctheta(t)*one_over_r(r)*vforce_buffer(PSI,vfp_p)

--- curl_viscous_pforce_theta ---
                qty(PSI) = -VFDBUFF(PSI,dvfp_p_dr) - one_over_r(r)*vforce_buffer(PSI,vfp_p) &
                + VFDBUFF(PSI,dvfp_r_dp)*csctheta(t)*one_over_r(r)

--- curl_viscous_pforce_phi ---
                qty(PSI) = VFDBUFF(PSI,dvfp_t_dr) + one_over_r(r)*vforce_buffer(PSI,vfp_t) &
                - VFDBUFF(PSI,dvfp_r_dt)*one_over_r(r)

--- curl_viscous_mforce_r ---
                qty(PSI) = VFDBUFF(PSI,dvfm_p_dt)*one_over_r(r) - VFDBUFF(PSI,dvfm_t_dp)*csctheta(t)*one_over_r(r) &
                + costheta(t)*csctheta(t)*one_over_r(r)*vforce_buffer(PSI,vfm_p)

--- curl_viscous_mforce_theta ---
                qty(PSI) = -VFDBUFF(PSI,dvfm_p_dr) - one_over_r(r)*vforce_buffer(PSI,vfm_p) &
                + VFDBUFF(PSI,dvfm_r_